# Bronze → Silver

Nesta etapa, vamos organizar os dados recebidos na Bronze, traduzir os nomes das colunas e aplicar os tratamentos previstos na atividade. Começaremos observando os dados para justificar cada decisão, depois construiremos e validaremos as tabelas Silver. As fontes Bronze serão apenas consultadas. A ingestão ainda possui 2.624 registros preservados somente nas auxiliares; por isso, os resultados desta versão serão limitados a cobertura das tabelas principais.

## Configuração e leitura das fontes

Vamos definir o catálogo e conferir se as seis tabelas Bronze estão disponíveis. Este notebook fará suas próprias leituras, sem depender das variáveis do notebook anterior. Os dados gravados em Delta continuam disponíveis mesmo após a desconexão da sessão.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalogo = "workspace"
schema_bronze = "bronze"
schema_silver = "silver"

spark.conf.set("spark.sql.session.timeZone", "UTC")

nomes_fontes = [
    "tb_movies_info",
    "tb_movies_financials",
    "tb_movies_metrics",
    "tb_movies_reviews",
    "tb_credits_and_tags",
    "tb_cotacao_dolar",
]

fontes_bronze = {}

for nome in nomes_fontes:
    caminho_tabela = f"{catalogo}.{schema_bronze}.{nome}"

    if not spark.catalog.tableExists(caminho_tabela):
        raise ValueError(f"Tabela não encontrada: {caminho_tabela}")

    fontes_bronze[nome] = spark.table(caminho_tabela)

print("As seis fontes Bronze estão disponíveis para leitura.")

As seis fontes Bronze estão disponíveis para leitura.


#### Resultado da conferência das fontes

As seis tabelas Bronze foram encontradas e estão disponíveis para leitura. Podemos continuar a preparação da Silver sem executar novamente o notebook de ingestão. Essa verificação confirma a disponibilidade das fontes; a qualidade e a estrutura dos dados serão avaliadas nas próximas etapas.

## Primeiro contato com as informações dos filmes

Vamos começar pela tabela que reúne os títulos, as datas de lançamento, a duração e o status dos filmes. Observaremos sua estrutura e uma pequena amostra. Na Bronze, esses campos foram mantidos como texto; na Silver, precisaremos interpretar seus valores antes de definir os tipos e aplicar as regras.

In [0]:
df_info_bronze = fontes_bronze["tb_movies_info"]

df_info_bronze.printSchema()

display(
    df_info_bronze
    .select(
        "id",
        "title",
        "release_date",
        "runtime",
        "original_language",
        "status",
        "ingestion_datetime",
    )
    .limit(10)
)

root
 |-- id: string (nullable = true)
 |-- tconst: string (nullable = true)
 |-- title: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- original_language: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- runtime: string (nullable = true)
 |-- status: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



id,title,release_date,runtime,original_language,status,ingestion_datetime
889699,holy spider,2022-07-13,118,fa,Released,2026-09-19T22:44:31.075Z
528773,"Tanguy, le retour",2019-04-10,0,fr,RELEASED,2026-09-19T22:44:31.075Z
375900,Tout schuss,2016-01-13,95,fr,Released,2026-09-19T22:44:31.075Z
888311,Voy a pasármelo bien,2022-08-05,108,es,Released,2026-09-19T22:44:31.075Z
539686,Shamshera,2022-07-22,158,hi,Released,2026-09-19T22:44:31.075Z
932339,Un novio para mi mujer,2022-07-22,92,es,Released,2026-09-19T22:44:31.075Z
426562,Magical Mystery or: The Return of Karl Schmidt,2017-08-31,111,de,Released,2026-09-19T22:44:31.075Z
426469,Growing Up Smith,2017-02-03,102,en,Released,2026-09-19T22:44:31.075Z
412489,Franca: Chaos and Creation,09-02-2016,80,en,Released,2026-09-19T22:44:31.075Z
518031,Seven Dinners,2019-02-14,91,ru,Released,2026-09-19T22:44:31.075Z


#### Resultado da observação inicial

A estrutura confirmou que as colunas originais estão como texto e ingestion_datetime está como timestamp. Na amostra, observamos datas com formatos diferentes, variações de letras no status e uma duração igual a zero. Esses exemplos orientam as próximas verificações, mas não representam uma contagem dos problemas da tabela inteira. A conversão dos tipos precisará considerar essas diferenças.

## Identificação dos filmes e registros repetidos

O documento exige um registro por filme na tabela de informações, mantendo a ingestão mais recente quando houver duplicações. Primeiro verificaremos se os identificadores estão preenchidos e quais aparecem mais de uma vez. Um identificador repetido não significa necessariamente que todas as informações sejam iguais. Por isso, observaremos os registros antes de escolher a versão que será mantida.

In [0]:
resumo_identificadores = df_info_bronze.agg(
    F.count("*").alias("total_registros"),
    F.countDistinct("id").alias("ids_distintos_nao_nulos"),
    F.count(
        F.when(F.col("id").isNull(), 1)
    ).alias("ids_nulos"),
    F.count(
        F.when(F.trim(F.col("id")) == "", 1)
    ).alias("ids_vazios_ou_espacos"),
)

display(resumo_identificadores)

df_ids_repetidos = (
    df_info_bronze
    .filter(
        F.col("id").isNotNull()
        & (F.trim(F.col("id")) != "")
    )
    .groupBy("id")
    .agg(F.count("*").alias("quantidade_registros"))
    .filter(F.col("quantidade_registros") > 1)
)

print("Identificadores repetidos:", df_ids_repetidos.count())

display(
    df_ids_repetidos
    .orderBy(F.desc("quantidade_registros"), "id")
    .limit(10)
)

# Mostramos os registros de até cinco identificadores repetidos.
ids_para_inspecao = (
    df_ids_repetidos
    .orderBy(F.desc("quantidade_registros"), "id")
    .limit(5)
    .select("id")
)

display(
    df_info_bronze
    .join(ids_para_inspecao, on="id", how="inner")
    .select(
        "id",
        "title",
        "release_date",
        "runtime",
        "status",
        "ingestion_datetime",
    )
    .orderBy("id", F.desc("ingestion_datetime"))
    .limit(30)
)

total_registros,ids_distintos_nao_nulos,ids_nulos,ids_vazios_ou_espacos
106585,97594,0,0


Identificadores repetidos: 5586


id,quantidade_registros
1506444,34
1305205,31
1309821,31
1336604,31
1433375,31
1611488,31
1310187,30
1314611,30
1383741,30
1408039,30


id,title,release_date,runtime,status,ingestion_datetime
1305205,DIE HART: DIE HARTER,30/05/2024,92,Released,2026-09-19T19:53:33.911Z
1305205,die hart: die harter,2024-05-30,92,Released,2026-09-19T19:53:33.911Z
1305205,Die Hart: Die Harter,2024-05-30,92,Released,2026-09-19T19:53:33.911Z
1305205,Die Hart: Die Harter,2024-05-30,92,Released,2026-09-19T19:53:33.911Z
1305205,Die Hart: Die Harter,2024-05-30,92,Released,2026-09-19T19:53:33.911Z
1305205,Die Hart: Die Harter,2024-05-30,92,Released,2026-09-19T19:53:33.911Z
1305205,Die Hart: Die Harter,2024-05-30,92,Released,2026-09-19T19:53:33.911Z
1305205,Die Hart: Die Harter,2024-05-30,92,RELEASED,2026-09-19T19:53:33.911Z
1305205,Die Hart: Die Harter,2024-05-30,92,Released,2026-09-19T19:53:33.911Z
1305205,Die Hart: Die Harter,30/05/2024,92,Released,2026-09-19T19:53:33.911Z


#### Resultado da verificação dos identificadores

Encontramos 106.585 registros e 97.594 identificadores distintos, sem identificadores nulos ou vazios. Há 5.586 identificadores repetidos, correspondendo a 8.991 registros excedentes se mantivermos uma linha por filme. 

A amostra mostrou versões do mesmo título com diferenças de capitalização, formato da data e status, mas com o mesmo horário de ingestão. Portanto, além de priorizar a ingestão mais recente, precisaremos verificar os empates antes de definir a deduplicação.

## Variações de status

Antes de traduzir o status, precisamos observar como ele aparece na origem. Vamos contar os valores originais e visualizar uma primeira padronização de espaços e letras. Essa comparação ajudará a identificar os ruídos que vão precisar ser removidos antes do mapeamento exigido pelo documento. Neste momento, ainda não substituiremos nenhum valor.

In [0]:
display(
    df_info_bronze
    .groupBy("status")
    .agg(F.count("*").alias("quantidade"))
    .withColumn(
        "status_sem_espacos_externos_em_minusculas",
        F.lower(F.trim(F.col("status"))),
    )
    .orderBy(F.desc("quantidade"), "status")
)

status,quantidade,status_sem_espacos_externos_em_minusculas
Released,80057,released
RELEASED,15986,released
released,9084,released
Post Production,530,post production
In Production,460,in production
IN PRODUCTION,105,in production
POST PRODUCTION,93,post production
post production,74,post production
in production,55,in production
In-Production,46,in-production


#### Resultado da inspeção dos status

Encontramos variações de letras maiúsculas e minúsculas, além de diferenças como In Production e In-Production. Esses valores precisam ser normalizados antes da tradução para que representem a mesma categoria. Também apareceram duas datas na coluna de status, indicando conteúdo incompatível com o campo. Conforme a atividade, valores que não puderem ser associados aos status previstos serão classificados como "Não Informado".

## Formatos das datas de lançamento

Uma data pode estar preenchida e ainda exigir uma interpretação diferente, dependendo da ordem de ano, mês e dia. Vamos agrupar os formatos aparentes sem converter os valores. Reconhecer o desenho de uma data não garante que ela exista no calendário, e valores como 03/04/2020 podem ser ambíguos (dependendo do padrão de datas escolhido).

In [0]:
data_texto = F.trim(F.col("release_date"))

df_datas_inspecao = df_info_bronze.select(
    "id",
    "release_date",
    F.when(
        F.col("release_date").isNull() | (data_texto == ""),
        "Ausente ou vazio",
    )
    .when(
        data_texto.rlike(r"^\d{4}-\d{2}-\d{2}$"),
        "AAAA-MM-DD",
    )
    .when(
        data_texto.rlike(r"^\d{2}-\d{2}-\d{4}$"),
        "Dois números e ano, separados por hífen",
    )
    .when(
        data_texto.rlike(r"^\d{2}/\d{2}/\d{4}$"),
        "Dois números e ano, separados por barra",
    )
    .when(
        data_texto.rlike(r"^\d{4}/\d{2}/\d{2}$"),
        "AAAA/MM/DD",
    )
    .otherwise("Outro formato")
    .alias("formato_aparente"),
)

display(
    df_datas_inspecao
    .groupBy("formato_aparente")
    .agg(F.count("*").alias("quantidade"))
    .orderBy(F.desc("quantidade"))
)

# Até cinco valores distintos de cada formato.
janela_exemplos = (
    Window
    .partitionBy("formato_aparente")
    .orderBy("release_date")
)

display(
    df_datas_inspecao
    .select("formato_aparente", "release_date")
    .distinct()
    .withColumn(
        "posicao",
        F.row_number().over(janela_exemplos),
    )
    .filter(F.col("posicao") <= 5)
    .drop("posicao")
    .orderBy("formato_aparente", "release_date")
)

formato_aparente,quantidade
AAAA-MM-DD,91307
"Dois números e ano, separados por barra",10574
"Dois números e ano, separados por hífen",4702
Outro formato,2


formato_aparente,release_date
AAAA-MM-DD,2016-01-01
AAAA-MM-DD,2016-01-02
AAAA-MM-DD,2016-01-03
AAAA-MM-DD,2016-01-04
AAAA-MM-DD,2016-01-05
"Dois números e ano, separados por barra",01/01/2016
"Dois números e ano, separados por barra",01/01/2017
"Dois números e ano, separados por barra",01/01/2018
"Dois números e ano, separados por barra",01/01/2019
"Dois números e ano, separados por barra",01/01/2020


#### Resultado da inspeção dos formatos de data

Encontramos 91.307 valores com aparência AAAA-MM-DD, 10.574 com dois componentes e ano separados por barras, 4.702 separados por hífens e dois textos incompatíveis com datas. Não apareceram valores no grupo de ausentes ou vazios. 

A amostra de duplicações mostrou 30/05/2024, confirmando a presença de dia/mês/ano com barras, mas exemplos como 01/01/2016 não permitem distinguir a ordem. Vamos verificar os componentes para orientar a interpretação dos formatos restantes.

## Identificando a ordem de dia e mês

Uma data como 01/02/2020 permite duas interpretações, mas um componente maior que 12 ajuda a identificar qual posição representa o dia. Vamos fazer essa comparação separadamente para datas com barras e hífens. Essa verificação identifica evidências sobre o formato, mas ainda não valida o calendário: valores como 31/02/2020 precisam ser rejeitados durante a conversão.

In [0]:
texto_data = F.trim(F.col("release_date"))

df_ordem_datas = (
    df_info_bronze
    .filter(
        texto_data.rlike(r"^\d{2}/\d{2}/\d{4}$")
        | texto_data.rlike(r"^\d{2}-\d{2}-\d{4}$")
    )
    .select(
        "id",
        "release_date",
        F.when(
            texto_data.contains("/"), "Barra"
        ).otherwise("Hífen").alias("separador"),
        F.substring(texto_data, 1, 2).cast("int").alias("primeiro"),
        F.substring(texto_data, 4, 2).cast("int").alias("segundo"),
    )
    .withColumn(
        "evidencia",
        F.when(
            F.col("primeiro").between(13, 31)
            & F.col("segundo").between(1, 12),
            "Indica dia-mês-ano",
        )
        .when(
            F.col("primeiro").between(1, 12)
            & F.col("segundo").between(13, 31),
            "Indica mês-dia-ano",
        )
        .when(
            F.col("primeiro").between(1, 12)
            & F.col("segundo").between(1, 12),
            "Permite as duas ordens",
        )
        .otherwise("Componentes incompatíveis"),
    )
)

display(
    df_ordem_datas
    .groupBy("separador", "evidencia")
    .agg(F.count("*").alias("quantidade"))
    .orderBy("separador", "evidencia")
)

janela_datas = (
    Window
    .partitionBy("separador", "evidencia")
    .orderBy("release_date")
)

display(
    df_ordem_datas
    .select("separador", "evidencia", "release_date")
    .distinct()
    .withColumn("posicao", F.row_number().over(janela_datas))
    .filter(F.col("posicao") <= 3)
    .drop("posicao")
    .orderBy("separador", "evidencia", "release_date")
)

separador,evidencia,quantidade
Barra,Indica dia-mês-ano,6197
Barra,Permite as duas ordens,4377
Hífen,Indica mês-dia-ano,2772
Hífen,Permite as duas ordens,1930


separador,evidencia,release_date
Barra,Indica dia-mês-ano,13/01/2016
Barra,Indica dia-mês-ano,13/01/2017
Barra,Indica dia-mês-ano,13/01/2018
Barra,Permite as duas ordens,01/01/2016
Barra,Permite as duas ordens,01/01/2017
Barra,Permite as duas ordens,01/01/2018
Hífen,Indica mês-dia-ano,01-13-2017
Hífen,Indica mês-dia-ano,01-13-2018
Hífen,Indica mês-dia-ano,01-13-2019
Hífen,Permite as duas ordens,01-01-2016


#### Resultado da verificação da ordem de dia e mês

Nas datas com barras, 6.197 registros indicam dia/mês/ano e 4.377 permitem as duas ordens. Nas datas com hífens e ano ao final, 2.772 indicam mês-dia-ano e 1.930 permitem as duas ordens. Não encontramos evidências da ordem contrária dentro desses grupos.

Por isso, adotaremos dia/mês/ano para barras e mês-dia-ano para hífens, além de ano-mês-dia para o formato iniciado pelo ano. Essa é uma convenção apoiada no padrão da atividade. A conversão também verificará se os valores existem no calendário.

## Conferindo possíveis deslocamentos de colunas

A inspeção encontrou dois textos na coluna de data e duas datas na coluna de status. Vamos observar os registros completos para verificar se esses problemas aparecem nas mesmas linhas e quais outros campos foram afetados. Não moveremos valores entre colunas apenas por parecerem compatíveis: uma correção desse tipo precisa de evidência sobre a posição original.

In [0]:
ids_datas_incompativeis = (
    df_datas_inspecao
    .filter(F.col("formato_aparente") == "Outro formato")
    .select("id")
)

ids_status_com_data = (
    df_info_bronze
    .filter(
        F.trim(F.col("status")).rlike(r"^\d{4}-\d{2}-\d{2}$")
    )
    .select("id")
)

ids_para_conferir = (
    ids_datas_incompativeis
    .unionByName(ids_status_com_data)
    .distinct()
)

# Incluímos todas as versões desses IDs para comparar os registros.
display(
    df_info_bronze
    .join(ids_para_conferir, on="id", how="inner")
    .orderBy("id", F.desc("ingestion_datetime"))
)

id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
482400,tt6286226,\Death,"Be Not Proud: The Making of \""\""The Exorcist III\""\""\""""",\Death,"Be Not Proud: The Making of \""\""The Exorcist III\""\""\""""",en,2016-10-25,Released,,2026-09-19T22:44:31.075Z
543670,tt7332514,\Ding Dong,"You're Dead! The Making of \""\""House\""\""\""""",\Ding Dong,"You're Dead! The Making of \""\""House\""\""\""""",en,2017-03-27,Released,,2026-09-19T22:44:31.075Z


#### Resultado da inspeção dos possíveis deslocamentos

Os problemas de data e status aparecem nos mesmos dois registros, de identificadores 482400 e 543670. As linhas apresentam trechos de títulos na coluna de data, o texto en na duração e datas na coluna de status, confirmando que a quantidade esperada de campos não garante seu posicionamento correto. Não apareceram outras versões desses identificadores na consulta. 

Não reconstruirei as linhas por suposição: os valores incompatíveis receberão o tratamento previsto para cada campo, e esses registros continuarão identificados como uma limitação de qualidade, inclusive nos campos textuais que a conversão de tipos não consegue validar.

## Empates entre as versões mais recentes

Como foi determinado manter a ingestão mais recente de cada filme, Vamos verificar quantos identificadores possuem mais de um registro nesse horário. Repetições com conteúdo idêntico não criam uma escolha entre informações diferentes; conteúdos diferentes no mesmo horário exigem um critério adicional de desempate. Essa distinção evita que a versão escolhida varie entre execuções sem uma justificativa.

In [0]:
janela_filme = Window.partitionBy("id")

df_versoes_mais_recentes = (
    df_info_bronze
    .withColumn(
        "ultima_ingestao_do_filme",
        F.max("ingestion_datetime").over(janela_filme),
    )
    .filter(
        F.col("ingestion_datetime")
        == F.col("ultima_ingestao_do_filme")
    )
    .drop("ultima_ingestao_do_filme")
)

# Comparamos as colunas originais, sem o horário de ingestão.
colunas_conteudo = [
    coluna
    for coluna in df_info_bronze.columns
    if coluna != "ingestion_datetime"
]

df_empates = (
    df_versoes_mais_recentes
    .groupBy("id")
    .agg(
        F.count("*").alias("registros_no_ultimo_horario"),
        F.countDistinct(
            F.struct(
                *[F.col(coluna) for coluna in colunas_conteudo]
            )
        ).alias("conteudos_distintos"),
    )
    .filter(F.col("registros_no_ultimo_horario") > 1)
    .withColumn(
        "tipo_empate",
        F.when(
            F.col("conteudos_distintos") == 1,
            "Conteúdo idêntico",
        ).otherwise("Conteúdos diferentes"),
    )
)

display(
    df_empates
    .groupBy("tipo_empate")
    .agg(
        F.count("*").alias("quantidade_filmes"),
        F.sum("registros_no_ultimo_horario").alias(
            "registros_envolvidos"
        ),
    )
    .orderBy("tipo_empate")
)

display(
    df_empates
    .filter(F.col("conteudos_distintos") > 1)
    .orderBy(F.desc("conteudos_distintos"), "id")
    .limit(10)
)

tipo_empate,quantidade_filmes,registros_envolvidos
Conteúdo idêntico,1656,3361
Conteúdos diferentes,3898,11140


id,registros_no_ultimo_horario,conteudos_distintos,tipo_empate
1355571,29,13,Conteúdos diferentes
1556716,29,12,Conteúdos diferentes
1695652,21,12,Conteúdos diferentes
1726387,28,12,Conteúdos diferentes
1375956,22,11,Conteúdos diferentes
1406124,19,11,Conteúdos diferentes
1506444,34,11,Conteúdos diferentes
1633007,20,11,Conteúdos diferentes
1216831,20,10,Conteúdos diferentes
1237950,21,10,Conteúdos diferentes


#### Resultado da verificação dos empates

Encontramos 1.656 filmes com repetições de conteúdo idêntico no horário mais recente e 3.898 com conteúdos diferentes nesse mesmo horário. Esses números consideram apenas as versões de última ingestão, enquanto a inspeção anterior considerava todas as versões disponíveis. 

Portanto, a data de ingestão continuará sendo o primeiro critério de escolha, mas não resolve todos os casos. Para os empates com conteúdos diferentes, vamos precisar de um critério explícito e reproduzível, sem apresentar a escolha como uma versão comprovadamente mais recente ou mais correta.

## Normalização e tradução dos status

Vamos remover espaços externos, padronizar as letras e tratar hífens e espaços repetidos antes de traduzir os status. O mapeamento seguirá as seis categorias exigidas pelo documento, mesmo que algumas não apareçam nesta execução. Valores não reconhecidos receberão "Não Informado". Manteremos o status original no DataFrame de trabalho para comparar o resultado, sem alterar a Bronze.

In [0]:
# Normalizamos somente a representação do status.
status_normalizado = F.lower(
    F.trim(
        F.regexp_replace(
            F.regexp_replace(
                F.col("status"),
                r"[-‐‑–—]+",
                " ",
            ),
            r"\s+",
            " ",
        )
    )
)

df_info_tratamento = df_info_bronze.withColumn(
    "status_normalizado",
    status_normalizado,
)

df_info_tratamento = df_info_tratamento.withColumn(
    "status_filme",
    F.when(
        F.col("status_normalizado") == "released",
        "Lançado",
    )
    .when(
        F.col("status_normalizado") == "post production",
        "Pós-Produção",
    )
    .when(
        F.col("status_normalizado") == "in production",
        "Em Produção",
    )
    .when(
        F.col("status_normalizado") == "planned",
        "Planejado",
    )
    .when(
        F.col("status_normalizado") == "rumored",
        "Rumores",
    )
    .when(
        F.col("status_normalizado") == "canceled",
        "Cancelado",
    )
    .otherwise("Não Informado"),
)

# Comparação entre a origem e o resultado do tratamento.
display(
    df_info_tratamento
    .groupBy(
        "status",
        "status_normalizado",
        "status_filme",
    )
    .agg(F.count("*").alias("quantidade"))
    .orderBy("status_filme", F.desc("quantidade"))
)

# Distribuição final, ainda antes da deduplicação dos filmes.
display(
    df_info_tratamento
    .groupBy("status_filme")
    .agg(F.count("*").alias("quantidade"))
    .orderBy(F.desc("quantidade"))
)

# Conferimos os registros que não puderam ser mapeados.
display(
    df_info_tratamento
    .filter(F.col("status_filme") == "Não Informado")
    .select(
        "id",
        "title",
        "status",
        "status_normalizado",
        "status_filme",
    )
    .orderBy("id")
    .limit(20)
)

status,status_normalizado,status_filme,quantidade
In Production,in production,Em Produção,460
IN PRODUCTION,in production,Em Produção,105
in production,in production,Em Produção,55
In-Production,in production,Em Produção,46
Released,released,Lançado,80057
RELEASED,released,Lançado,15986
released,released,Lançado,9084
2017-03-27,2017 03 27,Não Informado,1
2016-10-25,2016 10 25,Não Informado,1
Planned,planned,Planejado,42


status_filme,quantidade
Lançado,105127
Pós-Produção,742
Em Produção,666
Planejado,48
Não Informado,2


id,title,status,status_normalizado,status_filme
482400,\Death,2016-10-25,2016 10 25,Não Informado
543670,\Ding Dong,2017-03-27,2017 03 27,Não Informado


#### Resultado da normalização dos status

As diferenças de letras e hífens foram reunidas nas categorias previstas: 105.127 registros ficaram como "Lançado", 742 como "Pós-Produção", 666 como "Em Produção" e 48 como "Planejado". Apenas os dois registros com datas na coluna de status receberam "Não Informado", correspondendo aos deslocamentos identificados anteriormente. O tratamento manteve os 106.585 registros; essas contagens ainda incluem versões repetidas dos filmes.

## Conversão das datas de lançamento

Vamos converter as datas para o tipo DATE, usando os formatos identificados na inspeção: ano-mês-dia, dia/mês/ano e mês-dia-ano. Para os valores que permitem duas interpretações, seguiremos a convenção observada para cada separador. 

Primeiro organizaremos o texto no formato ano-mês-dia e depois faremos uma conversão segura, que retorna NULL quando a data não existe ou o conteúdo é incompatível. Manteremos o valor original para conferir as falhas e criaremos ano_lancamento a partir da data convertida. Datas futuras não serão removidas nesta etapa.

In [0]:
texto_data = F.trim(F.col("release_date"))

# Organizamos os formatos identificados como ano-mês-dia.
df_info_tratamento = df_info_tratamento.withColumn(
    "data_texto_padronizada",
    F.when(
        texto_data.rlike(r"^\d{4}-\d{2}-\d{2}$"),
        texto_data,
    )
    .when(
        texto_data.rlike(r"^\d{2}/\d{2}/\d{4}$"),
        F.concat(
            F.substring(texto_data, 7, 4),
            F.lit("-"),
            F.substring(texto_data, 4, 2),
            F.lit("-"),
            F.substring(texto_data, 1, 2),
        ),
    )
    .when(
        texto_data.rlike(r"^\d{2}-\d{2}-\d{4}$"),
        F.concat(
            F.substring(texto_data, 7, 4),
            F.lit("-"),
            F.substring(texto_data, 1, 2),
            F.lit("-"),
            F.substring(texto_data, 4, 2),
        ),
    )
    .otherwise(F.lit(None).cast("string")),
)

# A conversão também verifica a validade da data no calendário.
df_info_tratamento = (
    df_info_tratamento
    .withColumn(
        "data_lancamento",
        F.expr("try_cast(data_texto_padronizada AS DATE)"),
    )
    .withColumn(
        "ano_lancamento",
        F.year("data_lancamento"),
    )
    .withColumn(
        "resultado_conversao_data",
        F.when(
            F.col("release_date").isNull() | (texto_data == ""),
            "Ausente na origem",
        )
        .when(
            F.col("data_texto_padronizada").isNull(),
            "Formato não reconhecido",
        )
        .when(
            F.col("data_lancamento").isNull(),
            "Data inválida no calendário",
        )
        .otherwise("Convertida"),
    )
)

# Resumo de todas as conversões.
display(
    df_info_tratamento
    .groupBy("resultado_conversao_data")
    .agg(F.count("*").alias("quantidade"))
    .orderBy(F.desc("quantidade"))
)

# Exemplos dos três formatos identificados, antes e depois.
exemplos_datas = (
    df_info_tratamento
    .filter(F.col("data_lancamento").isNotNull())
    .withColumn(
        "formato_origem",
        F.when(
            texto_data.rlike(r"^\d{4}-\d{2}-\d{2}$"),
            "AAAA-MM-DD",
        )
        .when(
            texto_data.rlike(r"^\d{2}/\d{2}/\d{4}$"),
            "DD/MM/AAAA",
        )
        .otherwise("MM-DD-AAAA"),
    )
    .select(
        "formato_origem",
        "release_date",
        "data_lancamento",
        "ano_lancamento",
    )
    .distinct()
)

janela_amostra_datas = (
    Window
    .partitionBy("formato_origem")
    .orderBy("release_date")
)

display(
    exemplos_datas
    .withColumn(
        "posicao",
        F.row_number().over(janela_amostra_datas),
    )
    .filter(F.col("posicao") <= 3)
    .drop("posicao")
    .orderBy("formato_origem", "release_date")
)

# Conferimos os valores que não foram convertidos.
display(
    df_info_tratamento
    .filter(F.col("data_lancamento").isNull())
    .select(
        "id",
        "release_date",
        "data_texto_padronizada",
        "resultado_conversao_data",
    )
    .orderBy("id")
    .limit(20)
)

resultado_conversao_data,quantidade
Convertida,106583
Formato não reconhecido,2


formato_origem,release_date,data_lancamento,ano_lancamento
AAAA-MM-DD,2016-01-01,2016-01-01,2016
AAAA-MM-DD,2016-01-02,2016-01-02,2016
AAAA-MM-DD,2016-01-03,2016-01-03,2016
DD/MM/AAAA,01/01/2016,2016-01-01,2016
DD/MM/AAAA,01/01/2017,2017-01-01,2017
DD/MM/AAAA,01/01/2018,2018-01-01,2018
MM-DD-AAAA,01-01-2016,2016-01-01,2016
MM-DD-AAAA,01-01-2017,2017-01-01,2017
MM-DD-AAAA,01-01-2018,2018-01-01,2018


id,release_date,data_texto_padronizada,resultado_conversao_data
482400,"Be Not Proud: The Making of \""\""The Exorcist III\""\""\""""",null,Formato não reconhecido
543670,"You're Dead! The Making of \""\""House\""\""\""""",null,Formato não reconhecido


#### Resultado da conversão das datas

Foram convertidas 106.583 datas, utilizando as convenções definidas para os três formatos encontrados. Apenas os registros 482400 e 543670 permaneceram com data_lancamento e ano_lancamento nulos, pois apresentam trechos de títulos no campo de data. Não encontramos falhas de calendário entre os valores com formato reconhecido. A conversão manteve todos os registros e não tem objetivo de reconstruir os campos deslocados.

## Conversão da duração dos filmes

Vamos converter a duração para um número inteiro de minutos. Textos incompatíveis, valores negativos e números fora da capacidade do tipo INT serão tratados como ausentes. Essa validação de duração não negativa é uma decisão de implementação; a atividade exige a tipagem correta, mas não define uma regra específica para duração zero. Por isso, vamos manter os zeros e os apresentaremos separadamente, sem assumir que representam ausência de informação. O valor original continuará disponível para conferência.

In [0]:
texto_duracao = F.trim(F.col("runtime"))

# Só aceitamos representações de números inteiros não negativos.
# try_cast evita erro caso o número ultrapasse a capacidade de INT.
df_info_tratamento = df_info_tratamento.withColumn(
    "duracao_minutos",
    F.when(
        texto_duracao.rlike(r"^\d+$"),
        F.expr("try_cast(trim(runtime) AS INT)"),
    ).otherwise(F.lit(None).cast("int")),
)

df_info_tratamento = df_info_tratamento.withColumn(
    "resultado_conversao_duracao",
    F.when(
        F.col("runtime").isNull() | (texto_duracao == ""),
        "Ausente na origem",
    )
    .when(
        F.col("duracao_minutos").isNull(),
        "Valor incompatível",
    )
    .when(
        F.col("duracao_minutos") == 0,
        "Zero mantido",
    )
    .otherwise("Inteiro positivo"),
)

display(
    df_info_tratamento
    .groupBy("resultado_conversao_duracao")
    .agg(F.count("*").alias("quantidade"))
    .orderBy(F.desc("quantidade"))
)

# Agrupamos os valores rejeitados para entender os motivos.
display(
    df_info_tratamento
    .filter(
        F.col("resultado_conversao_duracao") == "Valor incompatível"
    )
    .groupBy("runtime")
    .agg(F.count("*").alias("quantidade"))
    .orderBy(F.desc("quantidade"), "runtime")
    .limit(20)
)

resultado_conversao_duracao,quantidade
Inteiro positivo,94716
Zero mantido,11867
Valor incompatível,2


runtime,quantidade
en,2


#### Resultado da conversão da duração

Foram identificadas 94.716 durações inteiras positivas e 11.867 durações iguais a zero, mantidas conforme a decisão desta etapa. Apenas dois valores foram considerados incompatíveis: ambos continham en, confirmando o deslocamento já observado nesses registros. Esses valores receberam NULL em duracao_minutos. Nenhum registro foi removido durante a conversão.

## Organização das colunas de informações dos filmes

Vamos selecionar os campos previstos no mapeamento da atividade e renomeá-los para português. O identificador permanecerá como texto, a data terá o tipo DATE, e duração e ano serão inteiros. Manteremos o horário da ingestão Bronze com o nome data_ingestao para aplicar a regra de escolha da versão mais recente. 

Não alteraremos a capitalização dos títulos, pois isso pode modificar a grafia de nomes próprios e siglas. Os campos auxiliares de investigação continuarão no DataFrame de trabalho, mas não serão incluídos nesta seleção.

In [0]:
df_info_preparada = df_info_tratamento.select(
    F.col("id").alias("id_filme"),
    F.col("title").alias("titulo"),
    F.col("original_title").alias("titulo_original"),
    "data_lancamento",
    "ano_lancamento",
    "duracao_minutos",
    F.col("original_language").alias("idioma_original"),
    "status_filme",
    F.col("overview").alias("sinopse"),
    F.col("tagline").alias("frase_divulgacao"),
    F.col("ingestion_datetime").alias("data_ingestao"),
)

df_info_preparada.printSchema()

display(
    df_info_preparada
    .select(
        "id_filme",
        "titulo",
        "data_lancamento",
        "ano_lancamento",
        "duracao_minutos",
        "status_filme",
    )
    .limit(10)
)

root
 |-- id_filme: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- titulo_original: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- ano_lancamento: integer (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = false)
 |-- sinopse: string (nullable = true)
 |-- frase_divulgacao: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)



id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,status_filme
293660,Deadpool,2016-02-09,2016,108,Lançado
299536,AVENGERS: INFINITY WAR,2018-04-25,2018,149,Lançado
299534,Avengers: Endgame,2019-04-24,2019,181,Lançado
475557,Joker,2019-10-01,2019,122,Lançado
271110,Captain America: Civil War,2016-04-27,2016,147,Lançado
284054,Black Panther,2018-02-13,2018,135,Lançado
284052,Doctor Strange,2016-10-25,2016,115,Lançado
315635,Spider-Man: Homecoming,2017-07-05,2017,133,Lançado
283995,Guardians of the Galaxy Vol. 2,2017-04-19,2017,137,Lançado
297761,Suicide Squad,2016-08-03,2016,123,Lançado


#### Resultado da organização das colunas

A estrutura apresentou os nomes em português e os tipos esperados: id_filme como texto, data_lancamento como DATE, ano_lancamento e duracao_minutos como inteiros e data_ingestao como timestamp. Os demais campos descritivos permaneceram como texto. 

A amostra também mostrou a conversão de 04-25-2018 para 2018-04-25. Essa seleção ainda contém as versões repetidas dos filmes, que serão tratadas a seguir.

## Escolha de uma versão por filme

Vamos manter uma linha por id_filme, priorizando a data de ingestão mais recente, conforme o requisito. Quando houver empate nesse horário, usaremos a ordenação dos campos tratados como critério de desempate, com valores nulos ao final. 

Essa escolha adicional não indica qual versão é mais verdadeira; ela apenas torna a seleção reproduzível. Se todas as informações tratadas forem iguais, qualquer uma dessas linhas produzirá o mesmo resultado. Manteremos a linha escolhida inteira, sem combinar campos de versões diferentes.

In [0]:
# A ingestão mais recente sempre tem prioridade.
# Os demais campos só participam quando o horário é igual.
colunas_desempate = [
    "titulo",
    "titulo_original",
    "data_lancamento",
    "ano_lancamento",
    "duracao_minutos",
    "idioma_original",
    "status_filme",
    "sinopse",
    "frase_divulgacao",
]

janela_deduplicacao = (
    Window
    .partitionBy("id_filme")
    .orderBy(
        F.col("data_ingestao").desc_nulls_last(),
        *[
            F.col(coluna).asc_nulls_last()
            for coluna in colunas_desempate
        ],
    )
)

df_info_filmes = (
    df_info_preparada
    .withColumn(
        "ordem_versao",
        F.row_number().over(janela_deduplicacao),
    )
    .filter(F.col("ordem_versao") == 1)
    .drop("ordem_versao")
)

# Verificamos quantidade, preenchimento e unicidade da chave.
total_antes = df_info_preparada.count()

conferencia = df_info_filmes.agg(
    F.count("*").alias("total_final"),
    F.countDistinct("id_filme").alias("ids_distintos"),
    F.count(
        F.when(
            F.col("id_filme").isNull()
            | (F.trim(F.col("id_filme")) == ""),
            1,
        )
    ).alias("ids_ausentes"),
    F.count(
        F.when(F.col("data_ingestao").isNull(), 1)
    ).alias("ingestoes_ausentes"),
).first()

if conferencia["ids_ausentes"] > 0:
    raise ValueError("Existem filmes sem identificador.")

if conferencia["ingestoes_ausentes"] > 0:
    raise ValueError("Existem filmes sem horário de ingestão.")

if conferencia["total_final"] != conferencia["ids_distintos"]:
    raise ValueError("Ainda existem identificadores repetidos.")

# Confirmamos que nenhum ID desapareceu.
ids_esperados = df_info_preparada.select("id_filme").distinct()
ids_obtidos = df_info_filmes.select("id_filme")

if ids_esperados.exceptAll(ids_obtidos).limit(1).count() > 0:
    raise ValueError("A deduplicação removeu algum identificador.")

# Confirmamos a prioridade da ingestão mais recente.
ultimas_ingestoes = (
    df_info_preparada
    .groupBy("id_filme")
    .agg(
        F.max("data_ingestao").alias("ingestao_esperada")
    )
)

versoes_incorretas = (
    df_info_filmes
    .join(ultimas_ingestoes, on="id_filme", how="inner")
    .filter(
        ~F.col("data_ingestao").eqNullSafe(
            F.col("ingestao_esperada")
        )
    )
)

if versoes_incorretas.limit(1).count() > 0:
    raise ValueError("Foi selecionada uma ingestão anterior à mais recente.")

display(
    spark.createDataFrame(
        [(
            total_antes,
            conferencia["total_final"],
            total_antes - conferencia["total_final"],
        )],
        [
            "registros_antes",
            "filmes_apos_deduplicacao",
            "versoes_excedentes_removidas",
        ],
    )
)

display(
    df_info_filmes
    .select(
        "id_filme",
        "titulo",
        "data_lancamento",
        "duracao_minutos",
        "status_filme",
        "data_ingestao",
    )
    .orderBy("id_filme")
    .limit(10)
)

registros_antes,filmes_apos_deduplicacao,versoes_excedentes_removidas
106585,97594,8991


id_filme,titulo,data_lancamento,duracao_minutos,status_filme,data_ingestao
1000004,Purple Beatz,2022-07-07,86,Lançado,2026-09-19T19:53:33.911Z
1000005,Aisha Brown: The First Black Woman Ever,2020-02-14,42,Lançado,2026-09-19T19:53:33.911Z
1000007,KYLE BROWNRIGG: INTRODUCING LYLE,2022-05-27,36,Lançado,2026-09-19T19:53:33.911Z
1000011,Worth Your Weight in Gold,2022-07-14,26,Lançado,2026-09-19T19:53:33.911Z
1000014,On va manquer !,2018-05-15,0,Lançado,2026-09-19T19:53:33.911Z
1000030,58 Hours: The Baby Jessica Story,2021-07-31,0,Lançado,2026-09-19T19:53:33.911Z
1000054,One Hundred Years and Hope,2022-06-18,107,Lançado,2026-09-19T19:53:33.911Z
1000058,Homecoming,2023-07-12,110,Lançado,2026-09-19T19:53:33.911Z
1000059,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,2016-04-05,116,Lançado,2026-09-19T19:53:33.911Z
1000073,A Chance To Win,2023-05-03,97,Lançado,2026-09-19T19:53:33.911Z


#### Resultado da deduplicação dos filmes

A tabela passou de 106.585 registros para 97.594 filmes, removendo 8.991 versões excedentes. As verificações confirmaram identificadores únicos e preenchidos, preservação de todos os identificadores e escolha da ingestão mais recente. Nos empates de horário, foi aplicado o critério de ordenação documentado, garantindo uma seleção reproduzível. O resultado está preparado no DataFrame df_info_filmes e ainda não foi gravado na Silver.

## Preparação das cotações do dólar

Vamos ler as cotações armazenadas na Bronze, renomear as colunas para português e converter os tipos. A data e hora da cotação representam quando o valor foi publicado pela fonte; a data de ingestão representa quando ele entrou no nosso projeto. Vamos conferir valores ausentes, taxas não positivas e eventuais versões diferentes da mesma publicação antes de organizar a série diária.

In [0]:
df_cotacao_bronze = fontes_bronze["tb_cotacao_dolar"]

df_cotacao_preparada = df_cotacao_bronze.select(
    F.col("dataHoraCotacao").alias("data_hora_original"),
    F.expr(
        "try_cast(dataHoraCotacao AS TIMESTAMP)"
    ).alias("data_hora_cotacao"),
    F.expr(
        "try_cast(cotacaoCompra AS DECIMAL(18,8))"
    ).alias("cotacao_compra"),
    F.col("ingestion_datetime").alias("data_ingestao"),
)

# Esses campos são necessários para organizar e utilizar a cotação.
cotacoes_invalidas = df_cotacao_preparada.filter(
    F.col("data_hora_cotacao").isNull()
    | F.col("cotacao_compra").isNull()
    | (F.col("cotacao_compra") <= 0)
    | F.col("data_ingestao").isNull()
)

if cotacoes_invalidas.limit(1).count() > 0:
    display(cotacoes_invalidas)
    raise ValueError(
        "Existem cotações inválidas. Revise os registros exibidos "
        "antes de continuar."
    )

df_cotacao_preparada = df_cotacao_preparada.drop(
    "data_hora_original"
)

if df_cotacao_preparada.limit(1).count() == 0:
    raise ValueError("A Bronze não possui cotações disponíveis.")

# No caso de republicação, priorizamos a ingestão mais recente.
janela_publicacao = Window.partitionBy("data_hora_cotacao")

df_cotacao_recente = (
    df_cotacao_preparada
    .withColumn(
        "ultima_ingestao",
        F.max("data_ingestao").over(janela_publicacao),
    )
    .filter(F.col("data_ingestao") == F.col("ultima_ingestao"))
    .drop("ultima_ingestao")
    .dropDuplicates()
)

# Não escolhemos arbitrariamente entre taxas conflitantes
# para a mesma publicação e o mesmo horário de ingestão.
conflitos_cotacao = (
    df_cotacao_recente
    .groupBy("data_hora_cotacao")
    .agg(
        F.countDistinct("cotacao_compra").alias("taxas_distintas")
    )
    .filter(F.col("taxas_distintas") > 1)
)

if conflitos_cotacao.limit(1).count() > 0:
    display(conflitos_cotacao)
    raise ValueError(
        "Existem taxas conflitantes na versão mais recente "
        "da mesma publicação."
    )

display(
    df_cotacao_recente.orderBy("data_hora_cotacao")
)

data_hora_cotacao,cotacao_compra,data_ingestao
2026-09-10T13:09:28.700Z,5.11430000,2026-09-19T19:54:16.522Z
2026-09-11T13:07:22.532Z,5.09120000,2026-09-19T19:54:16.522Z
2026-09-14T13:10:08.144Z,5.16900000,2026-09-19T19:54:16.522Z
2026-09-15T13:09:19.199Z,5.14840000,2026-09-19T19:54:16.522Z
2026-09-16T13:05:30.358Z,5.15200000,2026-09-19T19:54:16.522Z
2026-09-17T13:03:21.858Z,5.15150000,2026-09-19T21:54:14.253Z
2026-09-18T13:03:34.742Z,5.15690000,2026-09-19T21:54:14.253Z


#### Resultado da preparação das cotações

Foram preparadas sete publicações, entre 10/09/2026 e 18/09/2026. As verificações não encontraram datas inválidas, taxas ausentes ou não positivas, horários de ingestão ausentes ou conflitos entre as versões mais recentes da mesma publicação. As datas foram convertidas para timestamp e os valores mantidos como decimal.

## Construção da série diária de cotações

A atividade exige uma sequência contínua de dias, preenchendo as datas sem publicação com a última cotação disponível. Quando houver mais de uma publicação no mesmo dia, adotaremos a de horário mais recente como referência diária. Vamos criar o calendário desde a primeira cotação armazenada até uma data de referência informada no notebook, cujo padrão será o dia atual em Recife. Não preencheremos dias anteriores à primeira cotação conhecida nem usaremos publicações posteriores à data de referência. Manteremos a data de origem da taxa e um indicador para distinguir valores publicados de valores carregados do dia anterior.

In [0]:
from datetime import datetime
from zoneinfo import ZoneInfo

dbutils.widgets.text(
    "data_referencia",
    "",
    "Data de referência (AAAA-MM-DD)",
)

referencia_informada = dbutils.widgets.get(
    "data_referencia"
).strip()

hoje = datetime.now(ZoneInfo("America/Recife")).date()

if referencia_informada:
    data_referencia = datetime.strptime(
        referencia_informada,
        "%Y-%m-%d",
    ).date()

    if data_referencia.isoformat() != referencia_informada:
        raise ValueError("Use o formato AAAA-MM-DD.")
else:
    data_referencia = hoje

if data_referencia > hoje:
    raise ValueError("A data de referência não pode ser futura.")

df_cotacao_periodo = (
    df_cotacao_recente
    .withColumn(
        "data_cotacao",
        F.to_date("data_hora_cotacao"),
    )
    .filter(F.col("data_cotacao") <= F.lit(data_referencia))
)

primeira_data = (
    df_cotacao_periodo
    .agg(F.min("data_cotacao").alias("primeira_data"))
    .first()["primeira_data"]
)

if primeira_data is None:
    raise ValueError(
        "Não há cotação disponível até a data de referência."
    )

# Uma publicação por dia: a de horário mais recente.
janela_dia = (
    Window
    .partitionBy("data_cotacao")
    .orderBy(F.col("data_hora_cotacao").desc())
)

df_cotacao_diaria = (
    df_cotacao_periodo
    .withColumn("ordem", F.row_number().over(janela_dia))
    .filter(F.col("ordem") == 1)
    .select(
        "data_cotacao",
        "cotacao_compra",
        F.col("data_cotacao").alias("data_origem_cotacao"),
    )
)

# Calendário completo, incluindo finais de semana e feriados.
df_calendario = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(primeira_data),
                F.lit(data_referencia),
                F.expr("INTERVAL 1 DAY"),
            )
        ).alias("data_cotacao")
    )
)

janela_preenchimento = (
    Window
    .orderBy("data_cotacao")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_cotacao_silver = (
    df_calendario
    .join(df_cotacao_diaria, on="data_cotacao", how="left")
    .withColumn(
        "cotacao_preenchida",
        F.col("cotacao_compra").isNull(),
    )
    .withColumn(
        "cotacao_compra",
        F.last(
            "cotacao_compra",
            ignorenulls=True,
        ).over(janela_preenchimento),
    )
    .withColumn(
        "data_origem_cotacao",
        F.last(
            "data_origem_cotacao",
            ignorenulls=True,
        ).over(janela_preenchimento),
    )
)

# Conferimos a continuidade e a ausência de preenchimento pelo futuro.
dias_esperados = (data_referencia - primeira_data).days + 1

resumo_cotacoes = df_cotacao_silver.agg(
    F.count("*").alias("total_dias"),
    F.countDistinct("data_cotacao").alias("dias_distintos"),
    F.count(
        F.when(F.col("cotacao_preenchida"), 1)
    ).alias("dias_preenchidos"),
).first()

if (
    resumo_cotacoes["total_dias"] != dias_esperados
    or resumo_cotacoes["dias_distintos"] != dias_esperados
):
    raise ValueError("A série diária possui lacunas ou datas repetidas.")

problemas_preenchimento = df_cotacao_silver.filter(
    F.col("cotacao_compra").isNull()
    | F.col("data_origem_cotacao").isNull()
    | (F.col("data_origem_cotacao") > F.col("data_cotacao"))
)

if problemas_preenchimento.limit(1).count() > 0:
    raise ValueError("O preenchimento das cotações apresentou problemas.")

print("Data de referência:", data_referencia)
print("Total de dias:", resumo_cotacoes["total_dias"])
print("Dias preenchidos:", resumo_cotacoes["dias_preenchidos"])

display(df_cotacao_silver.orderBy("data_cotacao"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Data de referência: 2026-09-21
Total de dias: 12
Dias preenchidos: 5


data_cotacao,cotacao_compra,data_origem_cotacao,cotacao_preenchida
2026-09-10,5.11430000,2026-09-10,false
2026-09-11,5.09120000,2026-09-11,false
2026-09-12,5.09120000,2026-09-11,true
2026-09-13,5.09120000,2026-09-11,true
2026-09-14,5.16900000,2026-09-14,false
2026-09-15,5.14840000,2026-09-15,false
2026-09-16,5.15200000,2026-09-16,false
2026-09-17,5.15150000,2026-09-17,false
2026-09-18,5.15690000,2026-09-18,false
2026-09-19,5.15690000,2026-09-18,true


#### Resultado da construção da série diária

A série contém 11 dias consecutivos, de 10/09/2026 até a referência de 20/09/2026, sem datas repetidas ou lacunas. Quatro dias foram preenchidos: 12 e 13 de setembro receberam a cotação de 11/09, enquanto 19 e 20 receberam a de 18/09. A coluna data_origem_cotacao preserva a data da taxa utilizada, e cotacao_preenchida identifica os dias sem publicação armazenada. Na data de referência, a taxa disponível é de R$ 5,1569 por dólar, originada em 18/09/2026.

## Preparação dos dados financeiros

Vamos tratar orçamento e receita conforme a atividade: reconhecer ausências textuais, interpretar símbolos e separadores e converter os valores para decimal. Valores zerados ou negativos serão considerados ausentes. Antes da conversão, observaremos os formatos existentes para evitar que a remoção indiscriminada de pontuação altere os números. Também verificaremos os identificadores repetidos, pois a tabela financeira precisa permitir a associação com os filmes sem multiplicar os resultados.

In [0]:
df_financeiro_bronze = fontes_bronze["tb_movies_financials"]

display(
    df_financeiro_bronze.agg(
        F.count("*").alias("total_registros"),
        F.countDistinct("id").alias("ids_distintos"),
        F.count(
            F.when(
                F.col("id").isNull()
                | (F.trim(F.col("id")) == ""),
                1,
            )
        ).alias("ids_ausentes"),
    )
)

# Colocamos orçamento e receita no mesmo formato para inspecioná-los.
# Isso é apenas uma visão de trabalho; não altera a tabela Bronze.
df_valores_financeiros = (
    df_financeiro_bronze
    .select(
        "id",
        F.lit("orcamento").alias("campo"),
        F.col("budget").alias("valor_original"),
    )
    .unionByName(
        df_financeiro_bronze.select(
            "id",
            F.lit("receita").alias("campo"),
            F.col("revenue").alias("valor_original"),
        )
    )
)

texto_valor = F.trim(F.col("valor_original"))

df_formatos_financeiros = (
    df_valores_financeiros
    .withColumn(
        "formato_aparente",
        F.when(
            F.col("valor_original").isNull() | (texto_valor == ""),
            "Nulo ou vazio",
        )
        .when(
            F.lower(texto_valor).isin(
                "unknown", "não informado", "n/a"
            ),
            "Ausência textual conhecida",
        )
        .when(
            texto_valor.rlike(r"^[+-]?\d+$"),
            "Inteiro sem separador",
        )
        .when(
            texto_valor.rlike(r"^[+-]?\d+[.,]\d+$"),
            "Número com um separador",
        )
        .when(
            texto_valor.rlike(r"^[+-]?[0-9.,]+$"),
            "Número com vários separadores",
        )
        .otherwise("Símbolos, espaços internos ou outros textos"),
    )
)

display(
    df_formatos_financeiros
    .groupBy("campo", "formato_aparente")
    .agg(F.count("*").alias("quantidade"))
    .orderBy("campo", F.desc("quantidade"))
)

# Exemplos distintos por grupo, sem coletar a base inteira.
janela_formatos_financeiros = (
    Window
    .partitionBy("campo", "formato_aparente")
    .orderBy("valor_original")
)

display(
    df_formatos_financeiros
    .select("campo", "formato_aparente", "valor_original")
    .distinct()
    .withColumn(
        "posicao",
        F.row_number().over(janela_formatos_financeiros),
    )
    .filter(F.col("posicao") <= 5)
    .drop("posicao")
    .orderBy("campo", "formato_aparente", "valor_original")
)

# Distinguimos repetições idênticas de valores diferentes para o mesmo ID.
df_repeticoes_financeiras = (
    df_financeiro_bronze
    .groupBy("id")
    .agg(
        F.count("*").alias("quantidade_registros"),
        F.countDistinct(
            F.struct("budget", "revenue")
        ).alias("combinacoes_financeiras"),
    )
    .filter(F.col("quantidade_registros") > 1)
    .withColumn(
        "tipo_repeticao",
        F.when(
            F.col("combinacoes_financeiras") == 1,
            "Valores originais idênticos",
        ).otherwise("Valores originais diferentes"),
    )
)

display(
    df_repeticoes_financeiras
    .groupBy("tipo_repeticao")
    .agg(
        F.count("*").alias("quantidade_ids"),
        F.sum("quantidade_registros").alias("registros_envolvidos"),
    )
    .orderBy("tipo_repeticao")
)

total_registros,ids_distintos,ids_ausentes
106165,99006,0


campo,formato_aparente,quantidade
orcamento,Inteiro sem separador,97985
orcamento,Ausência textual conhecida,6253
orcamento,"Símbolos, espaços internos ou outros textos",1927
receita,Inteiro sem separador,96761
receita,Ausência textual conhecida,9404


campo,formato_aparente,valor_original
orcamento,Ausência textual conhecida,N/A
orcamento,Inteiro sem separador,0
orcamento,Inteiro sem separador,1
orcamento,Inteiro sem separador,10
orcamento,Inteiro sem separador,100
orcamento,Inteiro sem separador,1000
orcamento,"Símbolos, espaços internos ou outros textos",$ 1
orcamento,"Símbolos, espaços internos ou outros textos",$ 10
orcamento,"Símbolos, espaços internos ou outros textos",$ 100
orcamento,"Símbolos, espaços internos ou outros textos",$ 1000


tipo_repeticao,quantidade_ids,registros_envolvidos
Valores originais diferentes,1251,5476
Valores originais idênticos,2554,5488


#### Resultado da inspeção financeira

Encontramos 106.165 registros para 99.006 identificadores distintos, sem identificadores ausentes. O orçamento apresenta 6.253 marcadores de ausência e 1.927 valores no grupo de símbolos ou outros textos, cujos exemplos mostram o prefixo $. 

A receita apresenta 9.404 marcadores de ausência, e os demais valores foram classificados como inteiros, incluindo negativos. Entre os identificadores repetidos, 2.554 possuem valores originais idênticos e 1.251 apresentam diferenças. 

Vamos converter os valores antes de avaliar essas diferenças, pois representações como $ 100 e 100 podem corresponder ao mesmo número.

## Definição da cotação para os cálculos

A atividade exige a conversão dos valores para reais, mas não determina a data da taxa aplicada a cada filme. Nesta implementação, usaremos a cotação disponível na data de referência do notebook, mantendo a mesma referência para todos os filmes. Essa escolha representa uma conversão nessa data, não uma atualização histórica dos valores nem uma conversão na data de lançamento. Vamos preservar a taxa aplicada, a data de referência e a data de origem da cotação para permitir a conferência dos cálculos.

In [0]:
df_taxa_referencia = (
    df_cotacao_silver
    .filter(F.col("data_cotacao") == F.lit(data_referencia))
    .select(
        F.col("data_cotacao").alias("data_referencia_cambio"),
        "data_origem_cotacao",
        F.col("cotacao_compra").alias("taxa_cambio_aplicada"),
    )
)

# Uma única taxa evita multiplicar linhas ao associá-la aos filmes.
if df_taxa_referencia.count() != 1:
    raise ValueError(
        "É necessário ter exatamente uma cotação "
        "para a data de referência."
    )

display(df_taxa_referencia)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


data_referencia_cambio,data_origem_cotacao,taxa_cambio_aplicada
2026-09-21,2026-09-18,5.15690000


#### Resultado da definição da cotação

Foi selecionada uma única taxa de R$ 5,1569 por dólar para a referência de 20/09/2026. A publicação de origem é de 18/09/2026, carregada até a referência pelo preenchimento da série diária. Essa taxa será aplicada ao orçamento, à receita e ao lucro, mantendo suas datas de referência e origem para conferência.

## Conversão dos valores financeiros

A primeira tentativa mostrou que o orçamento também contém o prefixo USD e abreviações de magnitude, como 10.0K e 20.0M. Vamos remover os prefixos monetários reconhecidos e interpretar K como mil e M como milhão, aplicando o multiplicador antes da conversão final. Também aceitaremos números decimais com ponto. Os marcadores textuais de ausência serão convertidos para NULL, assim como valores zerados ou negativos, conforme o documento. Representações ainda não contempladas serão exibidas para revisão, sem remover pontuação ou letras indiscriminadamente.

In [0]:
# Reiniciamos a partir da Bronze para não depender da tentativa anterior.
df_financeiro_tratamento = df_financeiro_bronze

campos_financeiros = [
    ("budget", "orcamento_usd"),
    ("revenue", "receita_usd"),
]

campos_com_problemas = []

for origem, destino in campos_financeiros:
    texto = F.trim(F.col(origem))

    ausencia = (
        F.col(origem).isNull()
        | (texto == "")
        | F.lower(texto).isin("unknown", "não informado", "n/a")
    )

    coluna_texto = f"{destino}_texto"
    coluna_base = f"{destino}_base"
    coluna_multiplicador = f"{destino}_multiplicador"
    coluna_expandida = f"{destino}_expandido"
    coluna_convertida = f"{destino}_convertido"
    coluna_resultado = f"resultado_{destino}"

    # Removemos apenas os prefixos reconhecidos.
    df_financeiro_tratamento = (
        df_financeiro_tratamento
        .withColumn(
            coluna_texto,
            F.when(
                ausencia,
                F.lit(None).cast("string"),
            ).otherwise(
                F.trim(
                    F.regexp_replace(
                        F.upper(texto),
                        r"^(?:USD|US\$|\$)\s*",
                        "",
                    )
                )
            ),
        )
    )

    # Exemplos aceitos: 10000, -10000, 10.0K e 20.0M.
    formato_reconhecido = F.col(coluna_texto).rlike(
        r"^[+-]?\d+(?:\.\d+)?[KM]?$"
    )

    df_financeiro_tratamento = (
        df_financeiro_tratamento
        .withColumn(
            coluna_base,
            F.when(
                formato_reconhecido,
                F.regexp_replace(
                    F.col(coluna_texto),
                    r"[KM]$",
                    "",
                ),
            ),
        )
        .withColumn(
            coluna_multiplicador,
            F.when(
                F.col(coluna_texto).endswith("K"),
                F.lit(1000),
            )
            .when(
                F.col(coluna_texto).endswith("M"),
                F.lit(1000000),
            )
            .otherwise(F.lit(1)),
        )
        .withColumn(
            coluna_expandida,
            F.expr(
                f"try_cast({coluna_base} AS DECIMAL(24,8))"
            ) * F.col(coluna_multiplicador),
        )
        .withColumn(
            coluna_convertida,
            F.expr(
                f"try_cast({coluna_expandida} AS DECIMAL(18,2))"
            ),
        )
        .withColumn(
            coluna_resultado,
            F.when(ausencia, "Ausente na origem")
            .when(
                F.col(coluna_convertida).isNull(),
                "Formato não contemplado ou valor fora do tipo",
            )
            .when(
                F.col(coluna_expandida) <= 0,
                "Zero ou negativo convertido para NULL",
            )
            .when(
                F.col(coluna_convertida) <= 0,
                "Valor positivo inferior à precisão de centavos",
            )
            .otherwise("Valor positivo convertido"),
        )
        .withColumn(
            destino,
            F.when(
                F.col(coluna_convertida) > 0,
                F.col(coluna_convertida),
            ).otherwise(F.lit(None).cast("decimal(18,2)")),
        )
    )

    print(f"Resultado: {destino}")

    display(
        df_financeiro_tratamento
        .groupBy(coluna_resultado)
        .agg(F.count("*").alias("quantidade"))
        .orderBy(F.desc("quantidade"))
    )

    valores_para_revisar = df_financeiro_tratamento.filter(
        F.col(coluna_resultado).isin(
            "Formato não contemplado ou valor fora do tipo",
            "Valor positivo inferior à precisão de centavos",
        )
    )

    if valores_para_revisar.limit(1).count() > 0:
        campos_com_problemas.append(origem)

        display(
            valores_para_revisar
            .groupBy(origem, coluna_texto, coluna_resultado)
            .agg(F.count("*").alias("quantidade"))
            .orderBy(F.desc("quantidade"))
            .limit(20)
        )

# Avaliamos os dois campos antes de interromper.
if campos_com_problemas:
    raise ValueError(
        "Revise os valores exibidos nestes campos: "
        + ", ".join(campos_com_problemas)
        + ". Não execute os cálculos financeiros ainda."
    )

display(
    df_financeiro_tratamento
    .select(
        "id",
        "budget",
        "orcamento_usd",
        "revenue",
        "receita_usd",
    )
    .limit(10)
)

# Conferência específica das abreviações.
display(
    df_financeiro_tratamento
    .filter(F.col("orcamento_usd_texto").rlike(r"[KM]$"))
    .select("budget", "orcamento_usd")
    .distinct()
    .orderBy("budget")
    .limit(15)
)

Resultado: orcamento_usd


resultado_orcamento_usd,quantidade
Zero ou negativo convertido para NULL,91028
Valor positivo convertido,8884
Ausente na origem,6253


Resultado: receita_usd


resultado_receita_usd,quantidade
Zero ou negativo convertido para NULL,93179
Ausente na origem,9404
Valor positivo convertido,3582


id,budget,orcamento_usd,revenue,receita_usd
293660,58000000,58000000.00,Unknown,null
299536,300000000,300000000.00,2052415039,2052415039.00
299534,356000000,356000000.00,2800000000,2800000000.00
475557,55000000,55000000.00,1074458282,1074458282.00
271110,250000000,250000000.00,Não Informado,null
284054,200000000,200000000.00,1349926083,1349926083.00
284052,180000000,180000000.00,676343174,676343174.00
315635,175000000,175000000.00,880166924,880166924.00
283995,200000000,200000000.00,863756051,863756051.00
297761,175000000,175000000.00,746846894,746846894.00


budget,orcamento_usd
1.0M,1000000.00
1.1K,1100.00
1.1M,1100000.00
1.2K,1200.00
1.2M,1200000.00
1.3M,1300000.00
1.4M,1400000.00
1.5K,1500.00
1.5M,1500000.00
1.6M,1600000.00


#### Resultado da conversão financeira

A regra ampliada interpretou os prefixos monetários e as abreviações K e M sem deixar formatos pendentes. No orçamento, foram convertidos 8.884 valores positivos, reconhecidas 6.253 ausências textuais e transformados em NULL 91.028 valores zerados ou negativos. Na receita, foram convertidos 3.582 valores positivos, reconhecidas 9.404 ausências textuais e transformados em NULL 93.179 valores zerados ou negativos. A conferência mostrou exemplos como 1.1K convertido para 1.100,00 e 1.0M para 1.000.000,00. Essas contagens ainda incluem versões repetidas dos filmes.

## Cálculo dos valores em reais, lucro e margem

Vamos associar a taxa de referência aos valores financeiros e calcular orçamento e receita em reais. O lucro será a receita menos o orçamento, podendo ser negativo quando houver prejuízo. Adotaremos margem percentual como lucro dividido pela receita, multiplicado por 100; essa definição é uma decisão documentada, pois o enunciado não fornece a fórmula. Quando faltar um valor necessário, o resultado dependente permanecerá NULL, sem substituir ausência por zero. Os cálculos usarão valores decimais e os resultados monetários serão arredondados para duas casas ao final.

In [0]:
# df_taxa_referencia já foi validado para conter exatamente uma linha.
df_financeiro_calculos = (
    df_financeiro_tratamento
    .crossJoin(df_taxa_referencia)
    .withColumn(
        "lucro_usd",
        F.col("receita_usd") - F.col("orcamento_usd"),
    )
    .withColumn(
        "orcamento_brl",
        F.round(
            F.col("orcamento_usd") * F.col("taxa_cambio_aplicada"),
            2,
        ),
    )
    .withColumn(
        "receita_brl",
        F.round(
            F.col("receita_usd") * F.col("taxa_cambio_aplicada"),
            2,
        ),
    )
    .withColumn(
        "lucro_brl",
        F.round(
            F.col("lucro_usd") * F.col("taxa_cambio_aplicada"),
            2,
        ),
    )
    .withColumn(
        "margem_lucro_percentual",
        F.when(
            F.col("receita_usd") > 0,
            F.round(
                (F.col("lucro_usd") / F.col("receita_usd"))
                * F.lit(100),
                2,
            ),
        ),
    )
)

# Conferimos se a associação com a cotação preservou a quantidade de linhas.
total_antes_calculos = df_financeiro_tratamento.count()
total_depois_calculos = df_financeiro_calculos.count()

if total_antes_calculos != total_depois_calculos:
    raise ValueError("A associação da cotação alterou a quantidade de linhas.")

display(
    df_financeiro_calculos
    .select(
        "id",
        "orcamento_usd",
        "receita_usd",
        "lucro_usd",
        "taxa_cambio_aplicada",
        "orcamento_brl",
        "receita_brl",
        "lucro_brl",
        "margem_lucro_percentual",
    )
    .limit(10)
)

display(
    df_financeiro_calculos.agg(
        F.count("*").alias("total_registros"),
        F.count(
            F.when(F.col("lucro_usd").isNull(), 1)
        ).alias("registros_sem_lucro_calculavel"),
        F.count(
            F.when(F.col("lucro_usd") < 0, 1)
        ).alias("registros_com_prejuizo"),
        F.count(
            F.when(F.col("margem_lucro_percentual").isNull(), 1)
        ).alias("registros_sem_margem_calculavel"),
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


id,orcamento_usd,receita_usd,lucro_usd,taxa_cambio_aplicada,orcamento_brl,receita_brl,lucro_brl,margem_lucro_percentual
293660,58000000.00,null,null,5.15690000,299100200.00,null,null,null
299536,300000000.00,2052415039.00,1752415039.00,5.15690000,1547070000.00,10584099114.62,9037029114.62,85.38
299534,356000000.00,2800000000.00,2444000000.00,5.15690000,1835856400.00,14439320000.00,12603463600.00,87.29
475557,55000000.00,1074458282.00,1019458282.00,5.15690000,283629500.00,5540873914.45,5257244414.45,94.88
271110,250000000.00,null,null,5.15690000,1289225000.00,null,null,null
284054,200000000.00,1349926083.00,1149926083.00,5.15690000,1031380000.00,6961433817.42,5930053817.42,85.18
284052,180000000.00,676343174.00,496343174.00,5.15690000,928242000.00,3487834114.00,2559592114.00,73.39
315635,175000000.00,880166924.00,705166924.00,5.15690000,902457500.00,4538932810.38,3636475310.38,80.12
283995,200000000.00,863756051.00,663756051.00,5.15690000,1031380000.00,4454303579.40,3422923579.40,76.85
297761,175000000.00,746846894.00,571846894.00,5.15690000,902457500.00,3851414747.67,2948957247.67,76.57


total_registros,registros_sem_lucro_calculavel,registros_com_prejuizo,registros_sem_margem_calculavel
106165,104483,576,104483


#### Resultado dos cálculos financeiros

A associação com a cotação preservou os 106.165 registros, sem multiplicar linhas. O lucro e a margem puderam ser calculados em 1.682 registros; nos outros 104.483, faltava orçamento ou receita válido. Foram encontrados 576 registros com prejuízo, mantidos como resultados negativos válidos. A amostra mostrou a aplicação da taxa de 5,1569 (dólar) e a preservação de resultados ausentes quando faltavam informações. Essas quantidades representam registros antes da deduplicação, não filmes únicos.

## Seleção de uma versão financeira por filme

Vamos manter um registro financeiro por identificador para evitar que os relacionamentos da Gold multipliquem os valores. Essa é uma decisão de modelagem para atender ao resultado de um registro por filme na tabela fato. Priorizaremos a ingestão mais recente e, nos empates, ordenaremos orçamento e receita tratados, com valores nulos ao final. A linha escolhida será preservada inteira, sem combinar orçamento de uma versão com receita de outra. Não removeremos registros apenas porque seus valores financeiros estão ausentes.

In [0]:
# Padronizamos os valores monetários para o tipo previsto na Gold.
colunas_monetarias = [
    "orcamento_usd",
    "receita_usd",
    "lucro_usd",
    "orcamento_brl",
    "receita_brl",
    "lucro_brl",
]

df_financeiro_tipado = df_financeiro_calculos

for coluna in colunas_monetarias:
    coluna_temporaria = f"{coluna}_tipado"

    df_financeiro_tipado = df_financeiro_tipado.withColumn(
        coluna_temporaria,
        F.expr(f"try_cast({coluna} AS DECIMAL(18,2))"),
    )

    # Não permitimos que uma limitação do tipo apague valores calculados.
    fora_do_tipo = df_financeiro_tipado.filter(
        F.col(coluna).isNotNull()
        & F.col(coluna_temporaria).isNull()
    )

    if fora_do_tipo.limit(1).count() > 0:
        raise ValueError(
            f"Existem valores de {coluna} fora de DECIMAL(18,2)."
        )

    df_financeiro_tipado = (
        df_financeiro_tipado
        .drop(coluna)
        .withColumnRenamed(coluna_temporaria, coluna)
    )

df_financeiro_preparado = df_financeiro_tipado.select(
    F.col("id").alias("id_filme"),
    *colunas_monetarias,
    "margem_lucro_percentual",
    "taxa_cambio_aplicada",
    "data_referencia_cambio",
    "data_origem_cotacao",
    F.col("ingestion_datetime").alias("data_ingestao"),
)

janela_financeiro = (
    Window
    .partitionBy("id_filme")
    .orderBy(
        F.col("data_ingestao").desc_nulls_last(),
        F.col("orcamento_usd").asc_nulls_last(),
        F.col("receita_usd").asc_nulls_last(),
    )
)

df_financeiro_filmes = (
    df_financeiro_preparado
    .withColumn(
        "ordem_versao",
        F.row_number().over(janela_financeiro),
    )
    .filter(F.col("ordem_versao") == 1)
    .drop("ordem_versao")
)

resumo_financeiro = df_financeiro_filmes.agg(
    F.count("*").alias("total_final"),
    F.countDistinct("id_filme").alias("ids_distintos"),
    F.count(
        F.when(
            F.col("id_filme").isNull()
            | (F.trim(F.col("id_filme")) == ""),
            1,
        )
    ).alias("ids_ausentes"),
    F.count(
        F.when(F.col("data_ingestao").isNull(), 1)
    ).alias("ingestoes_ausentes"),
).first()

if (
    resumo_financeiro["ids_ausentes"] > 0
    or resumo_financeiro["ingestoes_ausentes"] > 0
):
    raise ValueError("Há registros sem identificador ou data de ingestão.")

if resumo_financeiro["total_final"] != resumo_financeiro["ids_distintos"]:
    raise ValueError("Ainda existem identificadores financeiros repetidos.")

ids_financeiros_esperados = (
    df_financeiro_preparado.select("id_filme").distinct()
)

if (
    ids_financeiros_esperados
    .exceptAll(df_financeiro_filmes.select("id_filme"))
    .limit(1)
    .count() > 0
):
    raise ValueError("Algum identificador foi perdido na deduplicação.")

ultimas_ingestoes_financeiras = (
    df_financeiro_preparado
    .groupBy("id_filme")
    .agg(F.max("data_ingestao").alias("ingestao_esperada"))
)

if (
    df_financeiro_filmes
    .join(ultimas_ingestoes_financeiras, "id_filme")
    .filter(
        ~F.col("data_ingestao").eqNullSafe(
            F.col("ingestao_esperada")
        )
    )
    .limit(1)
    .count() > 0
):
    raise ValueError("A deduplicação não preservou a ingestão mais recente.")

total_financeiro_antes = df_financeiro_preparado.count()

display(
    spark.createDataFrame(
        [(
            total_financeiro_antes,
            resumo_financeiro["total_final"],
            total_financeiro_antes - resumo_financeiro["total_final"],
        )],
        [
            "registros_antes",
            "filmes_apos_deduplicacao",
            "versoes_excedentes_removidas",
        ],
    )
)

df_financeiro_filmes.printSchema()

display(
    df_financeiro_filmes
    .orderBy("id_filme")
    .limit(10)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


registros_antes,filmes_apos_deduplicacao,versoes_excedentes_removidas
106165,99006,7159


root
 |-- id_filme: string (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- margem_lucro_percentual: decimal(26,2) (nullable = true)
 |-- taxa_cambio_aplicada: decimal(18,8) (nullable = true)
 |-- data_referencia_cambio: date (nullable = false)
 |-- data_origem_cotacao: date (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)



id_filme,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,margem_lucro_percentual,taxa_cambio_aplicada,data_referencia_cambio,data_origem_cotacao,data_ingestao
1000004,null,null,null,null,null,null,null,5.15690000,2026-09-21,2026-09-18,2026-09-19T19:53:48.943Z
1000005,null,null,null,null,null,null,null,5.15690000,2026-09-21,2026-09-18,2026-09-19T19:53:48.943Z
1000007,null,null,null,null,null,null,null,5.15690000,2026-09-21,2026-09-18,2026-09-19T19:53:48.943Z
1000011,null,null,null,null,null,null,null,5.15690000,2026-09-21,2026-09-18,2026-09-19T19:53:48.943Z
1000014,null,null,null,null,null,null,null,5.15690000,2026-09-21,2026-09-18,2026-09-19T19:53:48.943Z
1000030,null,null,null,null,null,null,null,5.15690000,2026-09-21,2026-09-18,2026-09-19T19:53:48.943Z
1000054,null,null,null,null,null,null,null,5.15690000,2026-09-21,2026-09-18,2026-09-19T19:53:48.943Z
1000058,4700000.00,null,null,24237430.00,null,null,null,5.15690000,2026-09-21,2026-09-18,2026-09-19T19:53:48.943Z
1000059,null,null,null,null,null,null,null,5.15690000,2026-09-21,2026-09-18,2026-09-19T19:53:48.943Z
1000073,6000000.00,null,null,30941400.00,null,null,null,5.15690000,2026-09-21,2026-09-18,2026-09-19T19:53:48.943Z


#### Resultado da preparação da tabela financeira

A tabela passou de 106.165 registros para 99.006 filmes, removendo 7.159 versões excedentes. As verificações confirmaram identificadores únicos, preservação dos identificadores e prioridade da ingestão mais recente. A amostra mantém valores financeiros ausentes quando não há informação válida para calculá-los, preservando a taxa de 5,1569, a referência de 20/09/2026 e a origem da cotação em 18/09/2026. O resultado permanece no DataFrame df_financeiro_filmes, sem gravação.

## Tratamento das métricas de engajamento

Vamos preparar popularidade, notas e contagens de votos com os nomes e tipos previstos na atividade. Na popularidade, interpretaremos a vírgula como separador decimal, seguindo exemplos da origem como "50,399"; essa convenção não será aplicada às contagens de votos. Textos incompatíveis serão convertidos para NULL, sem extrair números de frases deslocadas. Notas fora de 0 a 10 e popularidade ou votos negativos também receberão NULL; zeros serão mantidos. 

Para permitir relacionamentos sem multiplicar os filmes na Gold, manteremos uma versão por identificador, priorizando a ingestão mais recente e usando os valores tratados como desempate técnico. Apresentaremos as perdas de conversão e exemplos dos valores invalidados.

In [0]:
df_metricas_bronze = fontes_bronze["tb_movies_metrics"]
df_metricas_tratamento = df_metricas_bronze

configuracao_metricas = [
    ("popularity", "popularidade", "DOUBLE", False),
    ("vote_average", "nota_media_tmdb", "DOUBLE", True),
    ("vote_count", "qtd_votos_tmdb", "INT", False),
    ("averageRating", "nota_media_imdb", "DOUBLE", True),
    ("numVotes", "qtd_votos_imdb", "INT", False),
]

resumos_metricas = []
exemplos_metricas = []

for origem, destino, tipo, eh_nota in configuracao_metricas:
    texto = F.trim(F.col(origem))

    # A convenção de vírgula decimal foi observada na popularidade.
    if origem == "popularity":
        texto = F.regexp_replace(texto, ",", ".")

    padrao = (
        r"^[+-]?\d+$"
        if tipo == "INT"
        else r"^[+-]?\d+(?:\.\d+)?$"
    )

    coluna_texto = f"{destino}_texto"
    coluna_numero = f"{destino}_numero"

    df_metricas_tratamento = (
        df_metricas_tratamento
        .withColumn(coluna_texto, texto)
        .withColumn(
            coluna_numero,
            F.when(
                F.col(coluna_texto).rlike(padrao),
                F.expr(f"try_cast({coluna_texto} AS {tipo})"),
            ),
        )
    )

    numero = F.col(coluna_numero)
    valido = numero.isNotNull() & (numero >= 0)

    if eh_nota:
        valido = valido & (numero <= 10)
    elif tipo == "DOUBLE":
        # Evita aceitar infinito em caso de estouro da conversão.
        valido = valido & (numero < F.lit(float("inf")))

    df_metricas_tratamento = df_metricas_tratamento.withColumn(
        destino,
        F.when(valido, numero).otherwise(F.lit(None).cast(tipo)),
    )

    ausencia = (
        F.col(origem).isNull()
        | (F.trim(F.col(origem)) == "")
        | F.lower(F.trim(F.col(origem))).isin(
            "unknown", "não informado", "n/a"
        )
    )

    resumos_metricas.append(
        df_metricas_tratamento.agg(
            F.lit(destino).alias("campo"),
            F.count(F.when(ausencia, 1)).alias("ausencias_origem"),
            F.count(
                F.when(
                    ~ausencia & numero.isNull(), 1
                )
            ).alias("incompativeis_com_tipo"),
            F.count(
                F.when(
                    numero.isNotNull() & ~valido, 1
                )
            ).alias("fora_dos_limites"),
            F.count(destino).alias("valores_validos"),
        )
    )

    exemplos_metricas.append(
        df_metricas_tratamento
        .filter(~ausencia & F.col(destino).isNull())
        .select(
            F.lit(destino).alias("campo"),
            F.col(origem).alias("valor_original"),
        )
        .distinct()
        .orderBy("valor_original")
        .limit(5)
    )

df_resumo_metricas = resumos_metricas[0]
df_exemplos_metricas = exemplos_metricas[0]

for resumo, exemplos in zip(
    resumos_metricas[1:], exemplos_metricas[1:]
):
    df_resumo_metricas = df_resumo_metricas.unionByName(resumo)
    df_exemplos_metricas = df_exemplos_metricas.unionByName(exemplos)

display(df_resumo_metricas)
display(df_exemplos_metricas.orderBy("campo", "valor_original"))

nomes_metricas = [
    destino for _, destino, _, _ in configuracao_metricas
]

df_metricas_preparadas = df_metricas_tratamento.select(
    F.col("id").alias("id_filme"),
    *nomes_metricas,
    F.col("ingestion_datetime").alias("data_ingestao"),
)

if (
    df_metricas_preparadas
    .filter(
        F.col("id_filme").isNull()
        | (F.trim(F.col("id_filme")) == "")
        | F.col("data_ingestao").isNull()
    )
    .limit(1)
    .count() > 0
):
    raise ValueError("Métricas sem identificador ou data de ingestão.")

janela_metricas = (
    Window
    .partitionBy("id_filme")
    .orderBy(
        F.col("data_ingestao").desc(),
        *[
            F.col(coluna).asc_nulls_last()
            for coluna in nomes_metricas
        ],
    )
)

df_metricas_engajamento = (
    df_metricas_preparadas
    .withColumn("ordem", F.row_number().over(janela_metricas))
    .filter(F.col("ordem") == 1)
    .drop("ordem")
)

total_metricas_antes = df_metricas_preparadas.count()
total_metricas_final = df_metricas_engajamento.count()
ids_metricas_esperados = (
    df_metricas_preparadas.select("id_filme").distinct().count()
)

if total_metricas_final != ids_metricas_esperados:
    raise ValueError("A quantidade final não corresponde aos IDs da origem.")

display(
    spark.createDataFrame(
        [(
            total_metricas_antes,
            total_metricas_final,
            total_metricas_antes - total_metricas_final,
        )],
        [
            "registros_antes",
            "filmes_apos_deduplicacao",
            "versoes_excedentes_removidas",
        ],
    )
)

display(df_metricas_engajamento.orderBy("id_filme").limit(10))

campo,ausencias_origem,incompativeis_com_tipo,fora_dos_limites,valores_validos
popularidade,1,3513,0,102942
nota_media_tmdb,0,29,3114,103313
qtd_votos_tmdb,8452,29,0,97975
nota_media_imdb,11388,2368,0,92700
qtd_votos_imdb,8993,2712,0,94751


campo,valor_original
nota_media_imdb,"Alexis Alvarez"""
nota_media_imdb,"Amy Katherine Taylor"""
nota_media_imdb,"Brittany Nugent"""
nota_media_imdb,"Colin Rosenblum"""
nota_media_imdb,"Eugenio Martín"""
nota_media_tmdb,Andria Chamberlin
nota_media_tmdb,Dave Capdevielle
nota_media_tmdb,Dheeraj Rattan
nota_media_tmdb,Fadette Drouard
nota_media_tmdb,Francis Gasperini


registros_antes,filmes_apos_deduplicacao,versoes_excedentes_removidas
106456,98197,8259


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb,data_ingestao
1000004,1.132,0.0,0,6.8,27,2026-09-19T19:28:23.034Z
1000005,0.6,0.0,0,null,40,2026-09-19T19:28:23.034Z
1000007,0.6,0.0,0,4.7,10,2026-09-19T19:28:23.034Z
1000011,1.169,0.0,0,4.8,16,2026-09-19T19:28:23.034Z
1000014,0.6,0.0,0,7.2,15,2026-09-19T19:28:23.034Z
1000030,0.615,0.0,0,null,25,2026-09-19T19:28:23.034Z
1000054,0.6,0.0,0,5.9,10,2026-09-19T19:28:23.034Z
1000058,1.489,6.75,6,6.2,382,2026-09-19T19:28:23.034Z
1000059,0.6,0.0,0,7.7,25,2026-09-19T19:28:23.034Z
1000073,13.212,6.8,15,null,236,2026-09-19T19:28:23.034Z


#### Resultado do tratamento das métricas

A tabela passou de 106.456 registros para 98.197 filmes, removendo 8.259 versões excedentes. Antes da deduplicação, foram identificados valores incompatíveis com o tipo em popularidade (3.513), nota TMDB (29), votos TMDB (29), nota IMDb (2.368) e votos IMDb (2.712). Os exemplos exibidos incluem nomes, frases e caminhos de imagens, coerentes com os deslocamentos da origem. 

Também foram invalidadas 3.114 notas TMDB fora da faixa de 0 a 10. As ausências foram mantidas e os zeros preservados. Os exemplos confirmam parte dos motivos de rejeição, mas não representam uma revisão individual de todos os valores invalidados.

## Tratamento das avaliações dos usuários

Vamos converter as notas para número, mantendo apenas valores entre 0 e 10, e substituir comentários nulos, vazios ou compostos somente por espaços pelo texto "Sem comentário". Uma nota inválida ficará ausente, mas a avaliação não será descartada por esse motivo. 

A tabela poderá conter várias avaliações do mesmo filme: a unicidade será definida pela combinação de filme, usuário, nota e comentário após o tratamento. Quando essa combinação se repetir, manteremos o horário de ingestão mais recente. Essa escolha também reúne registros que se tornam iguais após a padronização.

In [0]:
df_avaliacoes_bronze = fontes_bronze["tb_movies_reviews"]

# Aceitamos notas inteiras ou decimais, com ponto ou vírgula decimal.
df_avaliacoes_tratamento = (
    df_avaliacoes_bronze
    .withColumn(
        "nota_texto",
        F.regexp_replace(F.trim(F.col("nota")), ",", "."),
    )
    .withColumn(
        "nota_convertida",
        F.when(
            F.col("nota_texto").rlike(r"^[+-]?\d+(?:\.\d+)?$"),
            F.expr("try_cast(nota_texto AS DOUBLE)"),
        ),
    )
    .withColumn(
        "nota_usuario",
        F.when(
            F.col("nota_convertida").between(0, 10),
            F.col("nota_convertida"),
        ).otherwise(F.lit(None).cast("double")),
    )
    .withColumn(
        "comentario_ausente",
        F.col("comentario").isNull()
        | F.col("comentario").rlike(r"^\s*$"),
    )
    .withColumn(
        "comentario_usuario",
        F.when(
            F.col("comentario_ausente"),
            "Sem comentário",
        ).otherwise(F.col("comentario")),
    )
)

display(
    df_avaliacoes_tratamento.agg(
        F.count("*").alias("total_registros"),
        F.count(
            F.when(F.col("nota_convertida").isNull(), 1)
        ).alias("notas_ausentes_ou_incompativeis"),
        F.count(
            F.when(
                F.col("nota_convertida").isNotNull()
                & ~F.col("nota_convertida").between(0, 10),
                1,
            )
        ).alias("notas_fora_da_faixa"),
        F.count(
            F.when(F.col("comentario_ausente"), 1)
        ).alias("comentarios_preenchidos"),
    )
)

display(
    df_avaliacoes_tratamento
    .filter(F.col("nota_usuario").isNull())
    .groupBy("nota")
    .agg(F.count("*").alias("quantidade"))
    .orderBy(F.desc("quantidade"))
    .limit(10)
)

df_avaliacoes_preparadas = df_avaliacoes_tratamento.select(
    F.col("id").alias("id_filme"),
    F.col("nome").alias("nome_usuario"),
    "nota_usuario",
    "comentario_usuario",
    F.col("ingestion_datetime").alias("data_ingestao"),
)

if (
    df_avaliacoes_preparadas
    .filter(
        F.col("id_filme").isNull()
        | (F.trim(F.col("id_filme")) == "")
        | F.col("data_ingestao").isNull()
    )
    .limit(1)
    .count() > 0
):
    raise ValueError("Avaliações sem identificador ou data de ingestão.")

chave_avaliacao = [
    "id_filme",
    "nome_usuario",
    "nota_usuario",
    "comentario_usuario",
]

df_avaliacoes_usuarios = (
    df_avaliacoes_preparadas
    .groupBy(*chave_avaliacao)
    .agg(F.max("data_ingestao").alias("data_ingestao"))
)

total_avaliacoes_antes = df_avaliacoes_preparadas.count()
total_avaliacoes_final = df_avaliacoes_usuarios.count()

display(
    spark.createDataFrame(
        [(
            total_avaliacoes_antes,
            total_avaliacoes_final,
            total_avaliacoes_antes - total_avaliacoes_final,
        )],
        [
            "registros_antes",
            "avaliacoes_apos_deduplicacao",
            "repeticoes_removidas",
        ],
    )
)

display(
    df_avaliacoes_usuarios
    .orderBy("id_filme", "nome_usuario", "nota_usuario")
    .limit(10)
)

total_registros,notas_ausentes_ou_incompativeis,notas_fora_da_faixa,comentarios_preenchidos
32412,1651,0,7776


nota,quantidade
,1651


registros_antes,avaliacoes_apos_deduplicacao,repeticoes_removidas
32412,32412,0


id_filme,nome_usuario,nota_usuario,comentario_usuario,data_ingestao
1000004,Leandro Araújo 732,0.1,"Que filme ruim, não assistam.",2026-09-19T19:54:12.131Z
1000014,Adriana Fernandes 259,3.3,"Ruim, não vale o tempo investido.",2026-09-19T19:54:12.131Z
1000058,Matheus Santos 431,6.8,Sem comentário,2026-09-19T19:54:12.131Z
1000094,Fábio Dias 200,5.4,"Mediano, tem seus momentos mas nada demais.",2026-09-19T19:54:12.131Z
1000096,Leandro Monteiro 550,5.3,Sem comentário,2026-09-19T19:54:12.131Z
1000099,Priscila Rocha 596,4.4,"Ruim, não vale o tempo investido.",2026-09-19T19:54:12.131Z
1000116,Daniel Ribeiro 144,null,Sem comentário,2026-09-19T19:54:12.131Z
1000116,Luciana Oliveira 263,1.9,Horrível! Perda de tempo.,2026-09-19T19:54:12.131Z
1000116,Mônica Teixeira 618,7.6,Muito bom! Vale a pena assistir.,2026-09-19T19:54:12.131Z
1000130,Paulo Alves,4.8,"Ruim, não vale o tempo investido.",2026-09-19T19:54:12.131Z


#### Resultado do tratamento das avaliações

Foram mantidas as 32.412 avaliações. Encontramos 1.651 notas sem valor preenchido, que permaneceram como NULL, e nenhuma nota numérica fora da faixa de 0 a 10. Foram preenchidos 7.776 comentários ausentes ou em branco com "Sem comentário". Não houve repetições na combinação de filme, usuário, nota e comentário após o tratamento. A amostra confirma que avaliações sem nota continuam disponíveis e que um mesmo filme pode possuir avaliações de usuários diferentes.

## Separação dos gêneros, pessoas e empresas

Vamos preparar os elementos das listas de gêneros, atores, diretores, roteiristas e produtoras, mantendo o identificador do filme e o tipo de entidade. Primeiro separaremos os valores por vírgula ou ponto e vírgula e removeremos apenas espaços externos. Essa separação ainda é uma preparação: fragmentos de textos deslocados também podem aparecer como elementos. Conferiremos os gêneros encontrados e exemplos de nomes antes de definir os filtros, evitando excluir nomes legítimos apenas por conterem números ou pontuação.

In [0]:
df_creditos_bronze = fontes_bronze["tb_credits_and_tags"]

campos_creditos = [
    ("genres", "Gênero"),
    ("cast", "Ator"),
    ("directors", "Diretor"),
    ("writers", "Roteirista"),
    ("production_companies", "Produtora"),
]

partes_creditos = []

for coluna, tipo in campos_creditos:
    partes_creditos.append(
        df_creditos_bronze.select(
            F.col("id").alias("id_filme"),
            F.lit(tipo).alias("tipo_entidade"),
            F.col(coluna).alias("lista_original"),
            F.col("ingestion_datetime").alias("data_ingestao"),
        )
    )

df_listas_creditos = partes_creditos[0]

for parte in partes_creditos[1:]:
    df_listas_creditos = df_listas_creditos.unionByName(parte)

df_creditos_elementos = (
    df_listas_creditos
    .withColumn(
        "elemento_original",
        F.explode_outer(
            F.split(F.col("lista_original"), r"[,;]")
        ),
    )
    .withColumn(
        "elemento",
        F.trim(F.col("elemento_original")),
    )
)

# Esta classificação ajuda na revisão; ainda não exclui elementos.
df_creditos_elementos = df_creditos_elementos.withColumn(
    "sinal_inspecao",
    F.when(
        F.col("elemento").isNull()
        | (F.col("elemento") == "")
        | F.lower(F.col("elemento")).isin(
            "[]", "null", "none", "unknown", "não informado", "n/a"
        ),
        "Vazio ou marcador de ausência",
    )
    .when(
        F.col("elemento").rlike(r"^[+-]?\d+(?:[.,]\d+)?$"),
        "Somente número",
    )
    .when(
        F.lower(F.col("elemento")).rlike(
            r"https?://|www\.|^/.*\.(jpg|jpeg|png|webp)$"
        ),
        "Link ou caminho de imagem",
    )
    .when(
        F.size(F.split(F.col("elemento"), r"\s+")) >= 12,
        "Texto longo para revisão",
    )
    .otherwise("Outros elementos"),
)

display(
    df_creditos_elementos
    .groupBy("tipo_entidade", "sinal_inspecao")
    .agg(F.count("*").alias("quantidade_elementos"))
    .orderBy("tipo_entidade", F.desc("quantidade_elementos"))
)

# Frequências dos elementos encontrados na coluna de gêneros.
display(
    df_creditos_elementos
    .filter(F.col("tipo_entidade") == "Gênero")
    .groupBy("elemento")
    .agg(F.count("*").alias("quantidade"))
    .orderBy(F.desc("quantidade"), "elemento")
    .limit(50)
)

# Exemplos de cada grupo para pessoas e empresas.
janela_exemplos_creditos = (
    Window
    .partitionBy("tipo_entidade", "sinal_inspecao")
    .orderBy("elemento")
)

display(
    df_creditos_elementos
    .filter(F.col("tipo_entidade") != "Gênero")
    .select("tipo_entidade", "sinal_inspecao", "elemento")
    .distinct()
    .withColumn(
        "posicao",
        F.row_number().over(janela_exemplos_creditos),
    )
    .filter(F.col("posicao") <= 3)
    .drop("posicao")
    .orderBy("tipo_entidade", "sinal_inspecao", "elemento")
)

tipo_entidade,sinal_inspecao,quantidade_elementos
Ator,Outros elementos,582328
Ator,Vazio ou marcador de ausência,26648
Ator,Somente número,2099
Ator,Link ou caminho de imagem,29
Ator,Texto longo para revisão,9
Diretor,Outros elementos,107733
Diretor,Vazio ou marcador de ausência,12595
Diretor,Link ou caminho de imagem,72
Diretor,Texto longo para revisão,48
Diretor,Somente número,46


elemento,quantidade
Drama,31957
Comedy,19519
Documentary,19468
,12053
Horror,10435
Thriller,10266
Romance,7270
Action,7261
Animation,4997
Crime,4721


tipo_entidade,sinal_inspecao,elemento
Ator,Link ou caminho de imagem,/1PVyiuL8YQ9IMhmJk2FkX9Qp4zF.jpg
Ator,Link ou caminho de imagem,/3oElPLWq47ZZo0PluBTKhXK94ra.jpg
Ator,Link ou caminho de imagem,/aDloQdsFgA6WhlFeoJej8rNxthA.jpg
Ator,Outros elementos,""""
Ator,Outros elementos,'Ana Ika
Ator,Outros elementos,'E-Gotti' Eric Johnson
Ator,Somente número,0.6
Ator,Somente número,0.634
Ator,Somente número,0.657
Ator,Texto longo para revisão,A 10-year-old boy doesn't love his parents. So they take him to see a therapist.


#### Resultado da preparação dos gêneros e entidades

A inspeção encontrou gêneros reconhecíveis e combinações separadas por barra vertical, como Comedy|Drama, além de valores vazios, números e caminhos de imagens. Nas pessoas e empresas, também apareceram frases deslocadas, mas há nomes legítimos com apóstrofos, números e símbolos. Portanto, incluiremos a barra vertical na separação dos gêneros e trataremos os nomes com regras específicas. Textos longos serão tratados como possíveis descrições por uma regra conservadora, com exceções para instituições identificado na revisão; esse critério pode excluir nomes legítimos e não constitui prova de invalidade.

## Construção dos gêneros e das entidades

Vamos separar os gêneros por vírgula, ponto e vírgula ou barra vertical e comparar os elementos com as categorias reconhecidas na inspeção. Para pessoas e empresas, removeremos espaços excedentes e resíduos de aspas duplas nas extremidades, preservando apóstrofos e pontuação interna. A capitalização será padronizada por fins de agrupamento. 

Elementos vazios, marcadores de ausência, números isolados e links serão rejeitados. Possíveis frases descritivas serão separadas para revisão, pois uma regra textual não consegue distinguir perfeitamente todos os nomes. As tabelas manterão o vínculo com o filme e não terão relacionamentos repetidos.

In [0]:
# Categorias reconhecidas na inspeção da fonte.
generos_reconhecidos = [
    "Action",
    "Adventure",
    "Animation",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Family",
    "Fantasy",
    "History",
    "Horror",
    "Music",
    "Mystery",
    "Romance",
    "Science Fiction",
    "TV Movie",
    "Thriller",
    "War",
    "Western",
]

df_dominio_generos = spark.createDataFrame(
    [
        (genero.lower(), genero)
        for genero in generos_reconhecidos
    ],
    ["chave_genero", "nome_genero"],
)

df_generos_classificados = (
    df_creditos_bronze
    .select(
        F.col("id").alias("id_filme"),
        F.explode_outer(
            F.split(F.col("genres"), r"[,;|]")
        ).alias("genero_original"),
        F.col("ingestion_datetime").alias("data_ingestao"),
    )
    .withColumn(
        "chave_genero",
        F.lower(
            F.trim(
                F.regexp_replace(
                    F.regexp_replace(
                        F.col("genero_original"),
                        r'^[\s"\\\[\]]+|[\s"\\\[\]]+$',
                        "",
                    ),
                    r"\s+",
                    " ",
                )
            )
        ),
    )
    .join(df_dominio_generos, on="chave_genero", how="left")
)

df_generos = (
    df_generos_classificados
    .filter(F.col("nome_genero").isNotNull())
    .groupBy("id_filme", "nome_genero")
    .agg(F.max("data_ingestao").alias("data_ingestao"))
)

display(
    df_generos_classificados
    .withColumn(
        "resultado",
        F.when(
            F.col("nome_genero").isNotNull(),
            "Gênero reconhecido",
        ).otherwise("Ausente ou fora do domínio"),
    )
    .groupBy("resultado")
    .agg(F.count("*").alias("quantidade_elementos"))
)

# Exemplos não reconhecidos, para conferir a regra de domínio.
display(
    df_generos_classificados
    .filter(F.col("nome_genero").isNull())
    .groupBy("genero_original")
    .agg(F.count("*").alias("quantidade"))
    .orderBy(F.desc("quantidade"))
    .limit(20)
)

display(
    df_generos
    .groupBy("nome_genero")
    .agg(F.count("*").alias("quantidade_filmes"))
    .orderBy(F.desc("quantidade_filmes"))
)

print("Vínculos únicos entre filmes e gêneros:", df_generos.count())

resultado,quantidade_elementos
Ausente ou fora do domínio,14476
Gênero reconhecido,153952


genero_original,quantidade
,12053
0.6,177
1.4,15
0.0286,14
0.0,12
in contrast to the public domain ballet music. This version eventually premiered at the Cannes Film Festival in 2023,9
/vH9NaIgNziDA4zbISUDtLEEq2t4.jpg,7
/sMu7FZYn0IoT3gq1dWEOm85nugU.jpg,7
"horror-comedy musical.\\""\""""",6
/8DII7cTfMjp89nvUPuhDFVzRYYj.jpg,4


nome_genero,quantidade_filmes
Drama,32489
Documentary,19116
Comedy,18741
Thriller,10372
Horror,9799
Romance,7687
Action,6086
Crime,4766
Animation,4500
TV Movie,4106


Vínculos únicos entre filmes e gêneros: 141356


In [0]:
# Reiniciamos a preparação a partir dos elementos da célula 69.
df_entidades_inicio = (
    df_creditos_elementos
    .filter(F.col("tipo_entidade") != "Gênero")
    .withColumn(
        "nome_limpo",
        F.trim(
            F.regexp_replace(
                F.regexp_replace(
                    F.col("elemento"),
                    r'^[\s"\\\[\]]+|[\s"\\\[\]]+$',
                    "",
                ),
                r"\s+",
                " ",
            )
        ),
    )
)

df_entidades_separacao = df_entidades_inicio.withColumn(
    "nome_para_separar",
    F.when(
        F.col("tipo_entidade") == "Produtora",
        F.regexp_replace(
            F.col("nome_limpo"),
            r"(?i)\s+in association with\s+",
            ";",
        ),
    ).otherwise(F.col("nome_limpo")),
)

# A expansão fica sozinha, sem trim ou outra função ao redor.
df_entidades_expandidas = df_entidades_separacao.select(
    "id_filme",
    "tipo_entidade",
    "lista_original",
    "elemento_original",
    "data_ingestao",
    F.explode_outer(
        F.split(F.col("nome_para_separar"), ";")
    ).alias("nome_separado"),
)

# Limpamos os espaços somente depois de expandir.
df_entidades_base = (
    df_entidades_expandidas
    .withColumn(
        "nome_limpo",
        F.trim(F.col("nome_separado")),
    )
    .drop("nome_separado")
)

df_entidades_base = df_entidades_base.withColumn(
    "nome_limpo",
    F.when(
        F.col("tipo_entidade") == "Produtora",
        F.trim(
            F.regexp_replace(
                F.col("nome_limpo"),
                r"(?i)\s*\(in association with\)\s*$",
                "",
            )
        ),
    ).otherwise(F.col("nome_limpo")),
)

nome = F.col("nome_limpo")
nome_minusculo = F.lower(nome)

# Exceções identificadas na revisão dos textos longos.
# A preservação considera o nome institucional apresentado pela fonte.
instituicoes_preservadas = [
    "escuela internacional de cine y televisión de san antonio de los baños",
    "escuela de imagen y sonido de vigo (eisv) - producciones vigo s.l.",
]

instituicao_preservada = (
    (F.col("tipo_entidade") == "Produtora")
    & nome_minusculo.isin(*instituicoes_preservadas)
)

# Trechos descritivos encontrados entre os elementos antes aceitos.
# Fragmentos identificados na inspeção dos dados.
# Usamos correspondência exata para não excluir nomes
# legítimos apenas por começarem com "A" ou conterem números.
fragmentos_observados = [
    "& extermination in an american city",
    "a desolate before the disappearance.",
    "a former president*",
    "a great male artist",
    "a home-brewed movie about craft beer",
    "a life and death intervention.",
    "1990s chris",
    "a land of war",
    "a latina immigrant",
    "a lie cannot live forever",
    "a long time ago",
    "a maverick with an aversion to lying",
    "a novelist lacking inspiration",
    "a philosophizing wedding photographer and the karate-practicing actress zhenya",
    "a puzzled driver",
    "a richly detailed examination of the nuts and bolts of moviemaking",
    "a sitting senator",
]

ausente = (
    nome.isNull()
    | (nome == "")
    | nome_minusculo.isin(
        "null", "none", "unknown", "não informado", "n/a"
    )
)

somente_numero = nome.rlike(r"^[+-]?\d+(?:[.,]\d+)?$")

link_ou_imagem = nome_minusculo.rlike(
    r"https?://|www\.|^/.*\.(jpg|jpeg|png|webp)$"
)

sem_letras = ~nome.rlike(r"\p{L}")

referencia_temporal = nome_minusculo.rlike(
    r"^(?:\d+(?:st|nd|rd|th)\s+century|\d{4}s)$"
)

quantidade_aberturas = (
    F.length(nome)
    - F.length(F.regexp_replace(nome, r"\(", ""))
)

quantidade_fechamentos = (
    F.length(nome)
    - F.length(F.regexp_replace(nome, r"\)", ""))
)

parenteses_incompletos = (
    quantidade_aberturas != quantidade_fechamentos
)

texto_longo = F.size(F.split(nome, r"\s+")) >= 12

expressao_descritiva = nome_minusculo.rlike(
    r"\b(?:"
    r"you win some|you lose most|"
    r"must resolve|hired a film crew|"
    r"a celebration of|a letter arrived|"
    r"is bewildered|all set against|"
    r"in association with"
    r")\b"
)

df_entidades_classificadas = (
    df_entidades_base
    .withColumn(
        "resultado_limpeza",
        F.when(ausente, "Excluído: ausência")
        .when(somente_numero, "Excluído: número isolado")
        .when(link_ou_imagem, "Excluído: link ou imagem")
        .when(sem_letras, "Excluído: sem letras")
        .when(
            referencia_temporal,
            "Excluído: referência temporal isolada",
        )
        .when(
            nome_minusculo.isin(*fragmentos_observados),
            "Excluído: fragmento descritivo observado",
        )
        .when(
            instituicao_preservada,
            "Aceito: instituição preservada na revisão",
        )
        .when(
            parenteses_incompletos,
            "Excluído por política: fragmento incompleto",
        )
        .when(
            texto_longo | expressao_descritiva,
            "Excluído por política: possível descrição",
        )
        .otherwise("Aceito pelos critérios atuais"),
    )
    .withColumn(
        "nome_entidade",
        F.initcap("nome_limpo"),
    )
)

# Mantemos o diagnóstico, incluindo os motivos de exclusão.
df_entidades_excluidas = (
    df_entidades_classificadas
    .filter(~F.col("resultado_limpeza").startswith("Aceito"))
    .select(
        "id_filme",
        "tipo_entidade",
        "lista_original",
        "elemento_original",
        "nome_limpo",
        "resultado_limpeza",
        "data_ingestao",
    )
)

df_pessoas_empresas = (
    df_entidades_classificadas
    .filter(F.col("resultado_limpeza").startswith("Aceito"))
    .groupBy("id_filme", "tipo_entidade", "nome_entidade")
    .agg(F.max("data_ingestao").alias("data_ingestao"))
)

display(
    df_entidades_classificadas
    .groupBy("tipo_entidade", "resultado_limpeza")
    .agg(F.count("*").alias("quantidade_elementos"))
    .orderBy("tipo_entidade", "resultado_limpeza")
)

# Conferimos os nomes envolvidos nas correções desta revisão.
nomes_para_conferir = [
    *instituicoes_preservadas,
    "20th century fox",
    "esoteric productions",
    "practical mullet",
    "illusion industries",
    "space theatre",
    "turbo panda productions",
]

display(
    df_entidades_classificadas
    .filter(
        F.lower("nome_limpo").isin(*nomes_para_conferir)
    )
    .select(
        "tipo_entidade",
        "nome_limpo",
        "nome_entidade",
        "resultado_limpeza",
    )
    .distinct()
    .orderBy("tipo_entidade", "nome_limpo")
)

# Amostra dos aceitos para identificar possíveis resíduos restantes.
janela_nomes = (
    Window
    .partitionBy("tipo_entidade")
    .orderBy("nome_entidade")
)

display(
    df_pessoas_empresas
    .select("tipo_entidade", "nome_entidade")
    .distinct()
    .withColumn("posicao", F.row_number().over(janela_nomes))
    .filter(F.col("posicao") <= 10)
    .drop("posicao")
    .orderBy("tipo_entidade", "nome_entidade")
)

display(
    df_pessoas_empresas
    .groupBy("tipo_entidade")
    .agg(F.count("*").alias("vinculos_com_filmes"))
    .orderBy("tipo_entidade")
)

# Confirmamos que os fragmentos identificados não chegaram
# à tabela final de pessoas e empresas.
fragmentos_restantes = (
    df_pessoas_empresas
    .filter(
        F.lower("nome_entidade").isin(*fragmentos_observados)
    )
    .count()
)

assert fragmentos_restantes == 0, (
    "Ainda existem fragmentos descritivos conhecidos na tabela final."
)

print(
    "Verificação concluída: nenhum dos fragmentos "
    "descritivos identificados permaneceu na tabela final."
)

tipo_entidade,resultado_limpeza,quantidade_elementos
Ator,Aceito pelos critérios atuais,582325
Ator,Excluído por política: fragmento incompleto,2
Ator,Excluído por política: possível descrição,9
Ator,Excluído: ausência,26649
Ator,Excluído: link ou imagem,29
Ator,Excluído: número isolado,2099
Diretor,Aceito pelos critérios atuais,107706
Diretor,Excluído por política: fragmento incompleto,3
Diretor,Excluído por política: possível descrição,48
Diretor,Excluído: ausência,12595


tipo_entidade,nome_limpo,nome_entidade,resultado_limpeza
Produtora,20th Century Fox,20th Century Fox,Aceito pelos critérios atuais
Produtora,Escuela Internacional de Cine y Televisión de San Antonio de los Baños,Escuela Internacional De Cine Y Televisión De San Antonio De Los Baños,Aceito: instituição preservada na revisão
Produtora,Escuela de Imagen y Sonido de Vigo (EISV) - Producciones Vigo s.l.,Escuela De Imagen Y Sonido De Vigo (eisv) - Producciones Vigo S.l.,Aceito: instituição preservada na revisão
Produtora,Esoteric Productions,Esoteric Productions,Aceito pelos critérios atuais
Produtora,Illusion Industries,Illusion Industries,Aceito pelos critérios atuais
Produtora,Practical Mullet,Practical Mullet,Aceito pelos critérios atuais
Produtora,Space Theatre,Space Theatre,Aceito pelos critérios atuais
Produtora,Turbo Panda Productions,Turbo Panda Productions,Aceito pelos critérios atuais


tipo_entidade,nome_entidade
Ator,'ana Ika
Ator,'e-gotti' Eric Johnson
Ator,'jeeva' Ravi
Ator,'meesai' Mohan
Ator,'meesai' Rajendran
Ator,'om' Rakesh Chaturvedi
Ator,'poo' Ram
Ator,'sunday Jeff' Silverman
Ator,2 Chainz
Ator,2 Kupzz


tipo_entidade,vinculos_com_filmes
Ator,545775
Diretor,101396
Produtora,118521
Roteirista,125960


Verificação concluída: nenhum dos fragmentos descritivos identificados permaneceu na tabela final.


#### Resultado da construção dos gêneros e entidades

Foram construídos 141.356 vínculos únicos entre filmes e gêneros e 891.652 vínculos entre filmes e pessoas ou empresas. A revisão corrigiu exclusões indevidas e identificou fragmentos descritivos que ainda eram aceitos como nomes. Esses fragmentos foram incorporados à regra de exclusão, e a verificação confirmou que nenhum deles permaneceu na tabela final. 

Os motivos de exclusão continuam disponíveis no DataFrame de diagnóstico durante a execução. As regras não comprovam a validade de todos os nomes e podem produzir exclusões indevidas; essa limitação permanece registrada.

## Validação conjunta das tabelas Silver

Vamos conferir as sete tabelas preparadas, a unicidade de suas chaves, o preenchimento dos identificadores e as principais regras de valores. Também verificaremos se os fragmentos descritivos identificados foram removidos e contaremos os identificadores sem correspondência nas informações dos filmes. Esses vínculos serão informados sem exclusão automática. A aprovação confirma o atendimento às verificações implementadas, mantendo documentadas as limitações de cobertura e interpretação dos dados.

In [0]:
validacao_silver_aprovada = False

tabelas_silver = {
    "tb_info_filmes": df_info_filmes,
    "tb_cotacao_dolar": df_cotacao_silver,
    "tb_financeiro_filmes": df_financeiro_filmes,
    "tb_metricas_engajamento": df_metricas_engajamento,
    "tb_avaliacoes_usuarios": df_avaliacoes_usuarios,
    "tb_generos": df_generos,
    "tb_pessoas_empresas": df_pessoas_empresas,
}

chaves_silver = {
    "tb_info_filmes": ["id_filme"],
    "tb_cotacao_dolar": ["data_cotacao"],
    "tb_financeiro_filmes": ["id_filme"],
    "tb_metricas_engajamento": ["id_filme"],
    "tb_avaliacoes_usuarios": [
        "id_filme",
        "nome_usuario",
        "nota_usuario",
        "comentario_usuario",
    ],
    "tb_generos": ["id_filme", "nome_genero"],
    "tb_pessoas_empresas": [
        "id_filme",
        "tipo_entidade",
        "nome_entidade",
    ],
}

ids_filmes = df_info_filmes.select("id_filme")
resumo_validacao = []
problemas = []

for nome_tabela, df in tabelas_silver.items():
    chaves = chaves_silver[nome_tabela]
    total = df.count()
    total_chaves = df.select(*chaves).distinct().count()
    repeticoes = total - total_chaves

    # Notas e nomes de usuários podem estar ausentes.
    # Aqui conferimos a identificação do filme ou do dia da cotação.
    coluna_identificacao = (
        "data_cotacao"
        if nome_tabela == "tb_cotacao_dolar"
        else "id_filme"
    )

    identificador = F.col(coluna_identificacao)

    identificacoes_ausentes = df.filter(
        identificador.isNull()
        | (F.trim(identificador.cast("string")) == "")
    ).count()

    if coluna_identificacao == "id_filme":
        ids_sem_info = (
            df.select("id_filme")
            .distinct()
            .join(ids_filmes, on="id_filme", how="left_anti")
            .count()
        )
    else:
        ids_sem_info = 0

    if repeticoes > 0 or identificacoes_ausentes > 0:
        problemas.append(
            f"{nome_tabela}: chave repetida ou identificação ausente."
        )

    resumo_validacao.append((
        nome_tabela,
        total,
        repeticoes,
        identificacoes_ausentes,
        ids_sem_info,
    ))

display(
    spark.createDataFrame(
        resumo_validacao,
        [
            "tabela",
            "total_registros",
            "repeticoes_da_chave",
            "identificacoes_ausentes",
            "ids_sem_correspondencia_em_info",
        ],
    )
)

# Verificações das principais regras de negócio.
regras = [
    (
        "Orçamento e receita positivos quando preenchidos",
        df_financeiro_filmes,
        (F.col("orcamento_usd") <= 0)
        | (F.col("receita_usd") <= 0),
    ),
    (
        "Notas de engajamento entre zero e dez",
        df_metricas_engajamento,
        ~F.col("nota_media_tmdb").between(0, 10)
        | ~F.col("nota_media_imdb").between(0, 10),
    ),
    (
        "Popularidade e votos não negativos",
        df_metricas_engajamento,
        (F.col("popularidade") < 0)
        | (F.col("qtd_votos_tmdb") < 0)
        | (F.col("qtd_votos_imdb") < 0),
    ),
    (
        "Notas dos usuários entre zero e dez",
        df_avaliacoes_usuarios,
        ~F.col("nota_usuario").between(0, 10),
    ),
    (
        "Comentários preenchidos",
        df_avaliacoes_usuarios,
        F.col("comentario_usuario").isNull()
        | F.col("comentario_usuario").rlike(r"^\s*$"),
    ),
    (
        "Cotação positiva e sem uso de publicação futura",
        df_cotacao_silver,
        F.col("cotacao_compra").isNull()
        | (F.col("cotacao_compra") <= 0)
        | F.col("data_origem_cotacao").isNull()
        | (F.col("data_origem_cotacao") > F.col("data_cotacao")),
    ),
    (
        "Tipos de entidade previstos",
        df_pessoas_empresas,
        F.col("tipo_entidade").isNull()
        | ~F.col("tipo_entidade").isin(
            "Ator", "Diretor", "Roteirista", "Produtora"
        ),
    ),
]

resumo_regras = []

for descricao, df, condicao_invalida in regras:
    quantidade = df.filter(condicao_invalida).count()
    resumo_regras.append((descricao, quantidade))

    if quantidade > 0:
        problemas.append(descricao)

display(
    spark.createDataFrame(
        resumo_regras,
        ["regra", "registros_em_desacordo"],
    )
)

# Conferência compacta dos tipos preparados.
display(
    spark.createDataFrame(
        [
            (nome_tabela, coluna, tipo)
            for nome_tabela, df in tabelas_silver.items()
            for coluna, tipo in df.dtypes
        ],
        ["tabela", "coluna", "tipo"],
    )
)

if problemas:
    raise ValueError(
        "Problemas encontrados: " + " | ".join(problemas)
    )

fragmentos_restantes = (
    df_pessoas_empresas
    .filter(F.lower("nome_entidade").isin(*fragmentos_observados))
    .count()
)

if fragmentos_restantes > 0:
    raise ValueError(
        "Ainda existem fragmentos descritivos conhecidos "
        "na tabela de pessoas e empresas."
    )

validacao_silver_aprovada = True

print(
    "As verificações implementadas passaram. "
    "As tabelas estão prontas para gravação, "
    "com as limitações documentadas no notebook."
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


tabela,total_registros,repeticoes_da_chave,identificacoes_ausentes,ids_sem_correspondencia_em_info
tb_info_filmes,97594,0,0,0
tb_cotacao_dolar,12,0,0,0
tb_financeiro_filmes,99006,0,0,1549
tb_metricas_engajamento,98197,0,0,1538
tb_avaliacoes_usuarios,32412,0,0,440
tb_generos,141356,0,0,1155
tb_pessoas_empresas,891652,0,0,1506


regra,registros_em_desacordo
Orçamento e receita positivos quando preenchidos,0
Notas de engajamento entre zero e dez,0
Popularidade e votos não negativos,0
Notas dos usuários entre zero e dez,0
Comentários preenchidos,0
Cotação positiva e sem uso de publicação futura,0
Tipos de entidade previstos,0


tabela,coluna,tipo
tb_info_filmes,id_filme,string
tb_info_filmes,titulo,string
tb_info_filmes,titulo_original,string
tb_info_filmes,data_lancamento,date
tb_info_filmes,ano_lancamento,int
tb_info_filmes,duracao_minutos,int
tb_info_filmes,idioma_original,string
tb_info_filmes,status_filme,string
tb_info_filmes,sinopse,string
tb_info_filmes,frase_divulgacao,string


As verificações implementadas passaram. As tabelas estão prontas para gravação, com as limitações documentadas no notebook.


#### Resultado da validação conjunta

As sete tabelas passaram pelas verificações de unicidade das chaves, preenchimento dos identificadores e regras numéricas aplicadas. Os tipos apresentados estão coerentes com os campos preparados. Foram encontrados identificadores sem correspondência em tb_info_filmes: 1.549 no financeiro, 1.538 nas métricas, 440 nas avaliações, 1.155 nos gêneros e 1.506 nas pessoas e empresas. 

Essas quantidades são de identificadores distintos por tabela e não devem ser somadas como filmes diferentes. Os registros foram preservados; a cobertura dos relacionamentos precisará ser considerada na Gold. A validação não comprova, por si só, a qualidade semântica dos nomes de entidades.

### Gravação das tabelas Silver

Vamos salvar as sete tabelas tratadas no schema Silver, em formato Delta. Como este notebook reconstrói as tabelas a partir da Bronze, cada execução substituirá o conteúdo anterior das tabelas Silver correspondentes. Assim, repetir a execução não acrescenta novamente os mesmos registros. Essa estratégia mantém a versão atual dos dados tratados; a Bronze continua sendo nossa fonte para reprocessamento.

In [0]:
if not globals().get("validacao_silver_aprovada", False):
    raise ValueError(
        "Execute a célula 76 e resolva as falhas antes de gravar."
    )

catalogo_destino = "workspace"
schema_destino = "silver"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{catalogo_destino}.{schema_destino}"
)

# Conferimos todas as tabelas antes de iniciar qualquer gravação.
tabelas_vazias = [
    nome_tabela
    for nome_tabela, df in tabelas_silver.items()
    if df.limit(1).count() == 0
]

if tabelas_vazias:
    raise ValueError(
        "Gravação interrompida: existem tabelas vazias: "
        + ", ".join(tabelas_vazias)
    )

for nome_tabela, df in tabelas_silver.items():
    destino = (
        f"{catalogo_destino}.{schema_destino}.{nome_tabela}"
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(destino)
    )

    print(f"Tabela gravada: {destino}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Tabela gravada: workspace.silver.tb_info_filmes
Tabela gravada: workspace.silver.tb_cotacao_dolar
Tabela gravada: workspace.silver.tb_financeiro_filmes
Tabela gravada: workspace.silver.tb_metricas_engajamento
Tabela gravada: workspace.silver.tb_avaliacoes_usuarios
Tabela gravada: workspace.silver.tb_generos
Tabela gravada: workspace.silver.tb_pessoas_empresas


In [0]:
resumo_gravacao = []
falhas_gravacao = []

for nome_tabela, df_preparado in tabelas_silver.items():
    destino = (
        f"{catalogo_destino}.{schema_destino}.{nome_tabela}"
    )

    df_salvo = spark.table(destino)

    estrutura_preparada = df_preparado.dtypes
    estrutura_salva = df_salvo.dtypes

    if estrutura_preparada != estrutura_salva:
        raise ValueError(
            f"Estrutura diferente da preparada: {destino}"
        )

    total_preparado = df_preparado.count()
    total_salvo = df_salvo.count()

    # exceptAll considera também a quantidade de repetições.
    registros_faltantes = (
        df_preparado.exceptAll(df_salvo).limit(1).count()
    )

    registros_extras = (
        df_salvo.exceptAll(df_preparado).limit(1).count()
    )

    conteudo_confere = (
        total_preparado == total_salvo
        and registros_faltantes == 0
        and registros_extras == 0
    )

    formato = (
        spark.sql(f"DESCRIBE DETAIL {destino}")
        .select("format")
        .first()["format"]
    )

    if not conteudo_confere or formato.lower() != "delta":
        falhas_gravacao.append(destino)

    resumo_gravacao.append((
        destino,
        total_preparado,
        total_salvo,
        formato,
        conteudo_confere,
    ))

display(
    spark.createDataFrame(
        resumo_gravacao,
        [
            "tabela",
            "registros_preparados",
            "registros_salvos",
            "formato",
            "conteudo_confere",
        ],
    )
)

if falhas_gravacao:
    raise ValueError(
        "Falha na conferência: " + ", ".join(falhas_gravacao)
    )

print(
    "Conferência concluída: as sete tabelas Delta salvas "
    "correspondem aos dados preparados."
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


tabela,registros_preparados,registros_salvos,formato,conteudo_confere
workspace.silver.tb_info_filmes,97594,97594,delta,true
workspace.silver.tb_cotacao_dolar,12,12,delta,true
workspace.silver.tb_financeiro_filmes,99006,99006,delta,true
workspace.silver.tb_metricas_engajamento,98197,98197,delta,true
workspace.silver.tb_avaliacoes_usuarios,32412,32412,delta,true
workspace.silver.tb_generos,141356,141356,delta,true
workspace.silver.tb_pessoas_empresas,891652,891652,delta,true


Conferência concluída: as sete tabelas Delta salvas correspondem aos dados preparados.


#### Resultado da gravação

As sete tabelas foram salvas em formato Delta no schema workspace.silver. A conferência confirmou que o conteúdo armazenado corresponde integralmente aos dados preparados, incluindo as quantidades de repetições. Foram gravados 97.594 filmes, 11 dias de cotação, 99.006 registros financeiros, 98.197 registros de engajamento, 32.412 avaliações, 141.356 vínculos entre filmes e gêneros e 891.652 vínculos entre filmes e pessoas ou empresas. 

Essa verificação confirma a integridade da gravação; as limitações de cobertura e limpeza documentadas anteriormente continuam válidas.

## Como executar novamente

Para reconstruir a Silver, se executa o notebook desde o início, mantendo a mesma data de referência do câmbio quando quisermos reproduzir o processamento anterior. Não precisamos executar a Bronze novamente se as tabelas já estiverem salvas e não houver novos dados para carregar. A gravação substitui o conteúdo das tabelas Silver, evitando acúmulo de registros entre execuções. 

As sete tabelas são gravadas separadamente: se houver uma interrupção, algumas poderão estar atualizadas e outras não. Nesse caso, vamos concluir uma nova execução e sua conferência antes de iniciar a Gold. Os resultados poderão mudar quando houver alterações na Bronze, nas regras de tratamento ou na data de referência.

## Aprendizados e limitações desta camada

Nesta camada, transformamos os dados recebidos em tabelas com nomes padronizados, tipos adequados e regras explícitas para ausências, valores inválidos e duplicidades. Aprendemos que uma leitura sem erro não garante que o conteúdo esteja na coluna correta, e que uma chave única não garante a qualidade de todos os seus atributos. 

Mantivemos os vínculos sem correspondência em tb_info_filmes identificados nas verificações, sem excluí-los; essa cobertura deverá ser considerada nas junções da Gold. A limpeza de pessoas e empresas combina regras gerais e fragmentos identificados na inspeção, mas não valida externamente cada nome e pode deixar resíduos ou excluir casos legítimos.

A Silver também trabalha com as tabelas principais da Bronze, sem recuperar os registros que permaneceram apenas nas auxiliares. Nos cálculos financeiros, usamos a cotação da data de referência, preenchida pela última cotação disponível quando necessário, e não uma cotação histórica do lançamento de cada filme.